# Glider Flight Data Analysis

## 1. Import Libraries

In [ ]:
# Objective 1.1 : Import the necessary libraries for data analysis and visualization Check

import os, glob, math, random, json, webbrowser, warnings
from pathlib import Path
from typing import Any, Dict, Tuple

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from datetime import datetime
from functools import partial
from concurrent.futures import ProcessPoolExecutor

import libigc
from libigc import Flight
from larus_to_df import Larus2Df
from regex import VERBOSE
from simplekml import Kml, Color, AltitudeMode
from tqdm import tqdm

import plotly.express as px, plotly.graph_objects as go, plotly.io as pio
from plotly.subplots import make_subplots
from plotly.colors import sample_colorscale
from dash import Dash, dcc, html, Input, Output

import pyproj
from pyproj import Transformer, network

from scipy.interpolate import UnivariateSpline, RegularGridInterpolator
from scipy.ndimage import gaussian_filter1d, gaussian_filter

from IPython.display import HTML, display
from sklearn.cluster import DBSCAN

pd.set_option("display.max_rows", None)

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message="DataFrameGroupBy.apply operated on the grouping columns"
)
warnings.simplefilter(action="ignore", category=FutureWarning)

import logging
logger = logging.getLogger("glider")
logger.setLevel(logging.INFO)

if not logger.handlers:
    fh = logging.FileHandler("errors.log", encoding="utf-8")
    fh.setLevel(logging.WARNING)
    fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
    logger.addHandler(fh)

from concurrent.futures import ProcessPoolExecutor



In [ ]:
# Objective 1.2 : File paths and Configuration of the code 
# Centralized configuration variables for the entire notebook, grouped by objective/section.
# Edit these values to change paths, parameters, or switches globally.

# Config: objective 2.1 (Data Paths)
MODE = 2  # 1: all, 2: igc only, 3: f37 only

RESULTS_DIR = Path("Results")
RESULTS_DIR.mkdir(exist_ok=True)

BASE_FOLDER = r"Raw_data"
F37_FOLDER = r"C:\Users\arshad\Desktop\04_Flight_data\Raw_data\f37" # use this line if F37 data is stored locally
#F37_FOLDER = os.path.join(BASE_FOLDER, "f37") # use this line if F37 data is stored in BASE_FOLDER
IGC_FOLDER = os.path.join(BASE_FOLDER, "igc")

# Config: objective 3.1 (Thermal Detection)
READ_COLS_F37 = [
    "flighttime", "time", "timestamp",
    "Lat", "Long", "pos DWN", "hour", "minute", "second", "nanoseconds",
    "track GNSS", "speed GNSS", "TAS", "pos DWN", "pos DWN_wgs84", "vario",
    "vario_wind_N", "vario_wind_E",
    "Air Density", "temp", "Magnetic Disturbance", "G_load", "turn rate"
]
g = 9.81; #m = 1.0
DEFAULT_WINDOW_SEC = 20
DEFAULT_PROGRESS_THRESH = 0.9
DEFAULT_MIN_DURATION = 30.0
DEFAULT_MIN_GAIN = 50.0

# Config: objective 3.2 (IGC/LARUS Matching)
SELECTED_DATE = "2025-06-09"
KML_OUT_DIR_IGC = r"Results/kml/igc_thermals"
KML_OUT_DIR_LARUS = r"Results/kml/larus_thermals"
EXPORT_KML = 0  # 0: disable KML exports
LARUS_DIR = r"Results/f37_valid_parquets"
LARUS_CANDIDATES = sorted(glob.glob(os.path.join(LARUS_DIR, f"{SELECTED_DATE} *.parquet")))
T_HALF = 600   # seconds (±10 min)
R_KM = 10      # km
WIN = 7        # smoothing window

# Config: objective 4.1 (Clustering)
R_MERGE_KM = 0.5
R_EARTH_KM = 6371.0

# Config: objective 5.1 (Modelling Directories)
RESULTS_DIR = Path("Results")
PLOTS_DIR = RESULTS_DIR / "plots"
DATAFRAMES_DIR = RESULTS_DIR / "dataframes"
SUMMARY_DIR = DATAFRAMES_DIR / "thermal_groups_summary"
THERMALLING_EFFICIENCY_DIR = DATAFRAMES_DIR / "thermalling efficiency"
PILOT_ID_COL               = "Filename"   # pilot identifier (per IGC/log file)
G0                         = 9.81         # [m/s^2], for altitude-equivalent efficiency
MIN_TURN_RADIUS_M              = 40.0     # min manoeuvrable turn radius [m]
THERMALLING_REFERENCE_TAS_MS   = 25.0     # reference circling TAS [m/s] for model gain

# Config: objective 5.1 (Modelling Parameters)
TE_MIN, TE_MAX = -5.0, 30.0  # TE-rate limits for colour mapping (Blocks 2–3)
TE_MIN_2, TE_MAX_2 = -10.0, 50.0  # extended TE-rate limits (Block 3 modelling of strength distributions)
CFD_COLORSCALE = [
    [0.00, "rgb(0,0,255)"],
    [0.25, "rgb(0,255,255)"],
    [0.27, "rgb(0,255,0)"],
    [0.34, "rgb(255,255,0)"],
    [1.00, "rgb(255,0,0)"]
]
RANK_COLORS = ["#d00000", "#ffcc00", "#0066ff"]  # ranking colours (Block 3)
RADIUS_RINGS_M = (100.0, 200.0, 300.0)          # rings used in Block 3
TICK_SPACING   = 200.0                          # axis tick spacing (Block 3)
NUM_SLICES              = 4                     # xy-slice config (Block 3)
SLICE_HALF_THICKNESS0   = 40.0
SLICE_HALF_THICKNESS_MAX= 200.0
MIN_POINTS_PER_SLICE    = 100
TOP_N_CIRCLES      = 30                         # used in Block 2
CIRCLE_TRIM_SECONDS = 10.0                      # used in Blocks 2–3

# Config: objective 4.3 (Modelling Switches)
RUN_MODELLING_LOOPS = 1      # 0: only selected thermal group; 1: loop over top-N groups
top_group = 25               # used when RUN_MODELLING_LOOPS == 0
# TOP_GROUP = int(cluster_strength_sorted.iloc[0]["thermal_group"])
N_TOP_GROUPS = 10            # set 10, 50, etc; None => all ranked

# per-plot switches (0 = skip, 1 = generate)
PLOT_DRIFT_TE_RATE                = 0
PLOT_XY_SLICES_TERATE             = 0
PLOT_MEAN_PLANE_XY_TERATE         = 0
PLOT_ROLLING_TE_PROFILES          = 0
PLOT_THERMAL_CYLINDER             = 0
PLOT_ANIMATED_CYLINDER_PATHS      = 0
PLOT_ANIMATED_CYLINDER_DRIFT_FREE = 0
PLOT_MODEL_COMPARISON             = 0
RUN_DASH_APP                      = 0
COMPUTE_THERMALLING_EFFICIENCY    = 1

VERBOSE = False          # set True if you want detailed logging
PLOT_LOG: list[str] = [] # will collect plot file paths per thermal group

## 2. Parse Data

In [ ]:
# Objective 2.1 : Parse and load glider flight data from .f37 and .igc files 

def load_all_f37_data(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.f37")))
    valid, bad, parsed = [], [], []
    for file in files:
        try:
            parsed.append(Larus2Df(file))
            valid.append(os.path.basename(file))
        except Exception as e:
            logger.exception("Error parsing %s", file)
            bad.append(os.path.basename(file))
    return parsed, valid, bad

def load_all_igc_data(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.igc")))
    parsed, bad = [], []
    for file in files:
        try:
            f = Flight.create_from_file(file)
            if f.valid:
                parsed.append(f)
            else:
                logger.error("Invalid %s: %s", file, f.notes)
                bad.append(os.path.basename(file))
        except Exception as e:
            logger.exception("Error parsing %s", file)
            bad.append(os.path.basename(file))
    return parsed, [os.path.basename(f) for f in files], bad

def load_data(BASE_FOLDER, mode=1):
    igc_p, igc_f, igc_b, f37_p, f37_f, f37_b = [], [], [], [], [], []

    F37_FOLDER = r"C:\Users\arshad\Desktop\04_Flight_data\Raw_data\f37"
    #F37_FOLDER = os.path.join(BASE_FOLDER, "f37")

    if MODE in (1, 3):  # load f37
        f37_p, f37_f, f37_b = load_all_f37_data(F37_FOLDER)

    if MODE in (1, 2):  # load igc
        IGC_FOLDER = os.path.join(BASE_FOLDER, "igc")
        igc_p, igc_f, igc_b = load_all_igc_data(IGC_FOLDER)

    return f37_p, f37_f, f37_b, igc_p, igc_f, igc_b

# === Main Function ===
f37_parsed, f37_files, f37_bad, igc_parsed, igc_files, igc_bad = load_data(BASE_FOLDER, MODE)

print("Loaded f37 files:")
for i, name in enumerate(f37_files, start=1):
    print(f"[{i}] {name}")

print("Loaded igc files:")
for i, name in enumerate(igc_files, start=1):
    print(f"[{i}] {name}")

if f37_bad:
    logger.warning("Corrupted or invalid f37 files: %s", f37_bad)
if igc_bad:
    logger.warning("Corrupted or invalid igc files: %s", igc_bad)

## 3. Detect Thermal Regions and Build Dataframe to Store Thermal Data

In [ ]:
# Objective 3.1 : detect thermal regions for LARUS flights and build f37_thermals_all.parquet
# 1. Detect thermal regions using progress < 0.9 & climbRate > 0
# 2. Filter out those with alt gain < 50 m
# 3. Save thermal dataframe, KML files (optional) and print summary table

# -------------------------
# Utilities
# -------------------------

def haversine_vec(lat1, lon1, lat2, lon2):
    R = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R*2*np.arctan2(np.sqrt(a), np.sqrt(1-a))

# -------------------------
# Convert raw df -> base df
# -------------------------

def f37_to_base_df(df):
    # minimal base columns (TAS already present; no alt interpolation)
    base = pd.DataFrame({
        "flighttime": df["flighttime"],
        "lat": df["Lat"],
        "lon": df["Long"],
        "alt": -df["pos DWN_wgs84"] if "pos DWN_wgs84" in df.columns else df["pos DWN"],
        "TAS": df["TAS"],
        "vario": df.get("vario", np.nan),
        "bearing": df.get("track GNSS", np.nan),
        "wind_N": df.get("vario_wind_N", np.nan),
        "wind_E": df.get("vario_wind_E", np.nan),
        "Air_Density": df.get("Air Density", np.nan),
        "temp": df.get("temp", np.nan),
        "Magnetic_Disturbance": df.get("Magnetic Disturbance", np.nan),
        "G_load": df.get("G_load", np.nan),
        "turn_rate": df.get("turn rate", np.nan) * (180.0 / np.pi),
        "time": df["time"],
        "timestamp": df["timestamp"]
    })

    # --- Total Energy (use filtered alt/TAS, but no interpolation) ---
    g = 9.81
    WIN = 7  # odd window; set to 1 to disable smoothing

    # light, robust smoothing (median -> mean) to reduce sensor noise
    alt_filt = (pd.to_numeric(base["alt"], errors="coerce")
                  .rolling(WIN, center=True, min_periods=1).median()
                  .rolling(WIN, center=True, min_periods=1).mean())
    tas_filt = (pd.to_numeric(base["TAS"], errors="coerce")
                  .rolling(WIN, center=True, min_periods=1).median()
                  .rolling(WIN, center=True, min_periods=1).mean())

    base["alt"] = alt_filt
    base["TAS"] = tas_filt

    # ensure strictly increasing time for numeric gradient
    base = base.sort_values("flighttime").reset_index(drop=True)
    flighttime2 = base["flighttime"] + base.groupby("flighttime").cumcount()*1e-2

    # numerical derivatives (no group split needed for single f37 flight)
    def _grad(y, t):
        yv = pd.to_numeric(y, errors="coerce").to_numpy()
        tv = pd.to_numeric(t, errors="coerce").to_numpy()
        if len(yv) < 2: 
            return np.full(len(yv), np.nan, dtype=float)
        yv = pd.Series(yv).interpolate("nearest", limit_direction="both").to_numpy()
        return np.gradient(yv, tv)

    dALT_dt = _grad(base["alt"], flighttime2)
    dTAS_dt = _grad(base["TAS"], flighttime2)

    # Total Energy and Total Energy rate (per unit mass)
    base["TotalEnergy"] = g*base["alt"] + 0.5*(base["TAS"]**2)
    TE_rate_raw = g*dALT_dt + base["TAS"]*dTAS_dt

    # simple instrument-like IIR smoothing for TE rate
    def _iir(y, t, tau=1.8):
        y = pd.to_numeric(pd.Series(y), errors="coerce").to_numpy()
        t = pd.to_numeric(pd.Series(t), errors="coerce").to_numpy()
        out = np.full(len(y), np.nan, dtype=float)
        if len(y) == 0: 
            return out
        out[0] = 0 if not np.isfinite(y[0]) else y[0]
        for k in range(1, len(y)):
            dt = t[k] - t[k-1]
            a  = 0 if not np.isfinite(dt) or dt <= 0 else (1 - np.exp(-dt/tau))
            val= y[k] if np.isfinite(y[k]) else out[k-1]
            out[k] = out[k-1] + a*(val - out[k-1])
        return out

    base["TotalEnergy_rate"] = _iir(np.clip(TE_rate_raw, -200, 300), flighttime2)

    return base



# -------------------------
# Thermal classifier
# -------------------------
def classify_thermals(df_base, window_sec=DEFAULT_WINDOW_SEC,
                      progress_thresh=DEFAULT_PROGRESS_THRESH,
                      min_duration=DEFAULT_MIN_DURATION,
                      min_gain=DEFAULT_MIN_GAIN, show_progress=False):
    n = len(df_base)
    if n < 3:
        df_base["label"] = []
        return df_base, pd.DataFrame(columns=list(df_base.columns)+["ThermalNumber"])
    ft, lat, lon, alt = df_base["flighttime"].values, df_base["lat"].values, df_base["lon"].values, df_base["alt"].values
    seg_dist_km = haversine_vec(lat[:-1], lon[:-1], lat[1:], lon[1:]) if n>1 else np.array([])
    cumsum_dist = np.concatenate([[0.0], np.cumsum(seg_dist_km)])
    j_indices = np.searchsorted(ft, ft+window_sec, side="right")-1
    j_indices = np.clip(j_indices, 0, n-1)
    labels = np.full(n, "Glide", dtype=object)
    iterator = tqdm(range(n-1), desc="Classifying thermals", disable=not show_progress)
    for i in iterator:
        j = j_indices[i]
        if j <= i: continue
        track_km = haversine_vec(lat[i], lon[i], lat[j], lon[j])
        ground_km = cumsum_dist[j]-cumsum_dist[i]
        progress = (track_km/ground_km) if ground_km>0 else 1.0
        climb_rate = (alt[j]-alt[i])/(ft[j]-ft[i]+1e-6)
        if progress<=progress_thresh and climb_rate>0 : labels[i]="Thermal" 
        elif progress<=progress_thresh and climb_rate<=0 : labels[i]="EmergencyDescent" 
    df_base["label"] = labels
    thermals_list, thermal_counter, in_th = [], 1, False
    for idx, lab in enumerate(labels):
        if lab=="Thermal" and not in_th: in_th, start = True, idx
        elif lab!="Thermal" and in_th:
            in_th, end = False, idx-1
            dur, gain = ft[end]-ft[start], alt[end]-alt[start]
            if dur>=min_duration and gain>=min_gain:
                seg=df_base.iloc[start:end+1].copy()
                seg["ThermalNumber"]=thermal_counter
                thermals_list.append(seg); thermal_counter+=1
    if in_th:
        end = n-1; dur, gain = ft[end]-ft[start], alt[end]-alt[start]
        if dur>=min_duration and gain>=min_gain:
            seg=df_base.iloc[start:end+1].copy()
            seg["ThermalNumber"]=thermal_counter
            thermals_list.append(seg)
    thermals_df = pd.concat(thermals_list, ignore_index=True) if thermals_list else pd.DataFrame(columns=list(df_base.columns)+["ThermalNumber"])
    return df_base, thermals_df

# -------------------------
# KML exporter
# -------------------------
def export_kml(fname, df_base, thermals_df):
    kml = Kml()
    kml.newlinestring(name="Full Flight Path", coords=list(zip(df_base.lon, df_base.lat, df_base.alt))).style.linestyle.color = Color.gray
    for th_num, th in thermals_df.groupby("ThermalNumber"):
        coords = list(zip(th.lon, th.lat, th.alt))
        ls = kml.newlinestring(name=f"Thermal {int(th_num)}", coords=coords)
        ls.style.linestyle.width=3
        ls.style.linestyle.color=Color.rgb(random.randint(0,255),random.randint(0,255),random.randint(0,255),a=200)
        ls.altitudemode = AltitudeMode.absolute
        dur = th.flighttime.iloc[-1]-th.flighttime.iloc[0]
        alt_gain = th.alt.iloc[-1]-th.alt.iloc[0]
        ls.description=f"Duration: {dur:.1f}s<br>Δalt: {alt_gain:.1f} m"
    os.makedirs(os.path.dirname(fname), exist_ok=True)
    kml.save(fname)

# -------------------------
# Per-file processing
# -------------------------
def process_flight(fpath, out_dir="output", show_progress=True):
    fname = os.path.basename(fpath)
    if show_progress:
        print(f"Processing: {fname}")

    df = pd.read_parquet(fpath, columns=READ_COLS_F37)
    df = df.drop_duplicates()
    df_base = f37_to_base_df(df)

    # classify thermals
    df_cls, thermals_before = classify_thermals(df_base, show_progress=show_progress)

    # assign thermal_id
    if EXPORT_KML and not thermals_before.empty:
        thermal_map = thermals_before[["flighttime", "ThermalNumber"]].drop_duplicates("flighttime")
        df_cls = df_cls.merge(thermal_map, on="flighttime", how="left")
        df_cls["thermal_id"] = df_cls["ThermalNumber"].fillna(-1).astype(int)
        df_cls.drop(columns=["ThermalNumber"], inplace=True, errors="ignore")
        thermals_before["Filename"] = fname   # ✅ always keep filename
    else:
        df_cls["thermal_id"] = -1
        thermals_before = pd.DataFrame(columns=list(df_base.columns) + ["ThermalNumber", "Filename"])

    # export KMLs (only when thermals exist)
    if not thermals_before.empty:
        os.makedirs(out_dir, exist_ok=True)
        export_kml(os.path.join(out_dir, fname.replace(".parquet", "_thermals.kml")),
                   df_base, thermals_before)

    return fname, df_base, df_cls, thermals_before


# -------------------------
# Batch processing
# -------------------------
def process_all(f37_files, out_dir="output", show_progress=True):
    results, thermals_all_before = [], []
    for i, f in enumerate(f37_files, 1):
        fname, df_base, df_cls, thermals_before = process_flight(
            f, out_dir, show_progress=show_progress
        )
        results.append((fname, df_base, df_cls, thermals_before))
        if not thermals_before.empty:
            thermals_all_before.append(thermals_before)

    thermals_all_before_df = pd.concat(thermals_all_before, ignore_index=True) if thermals_all_before else pd.DataFrame()
    return results, thermals_all_before_df


# -------------------------
# Main
# -------------------------
input_dir = LARUS_DIR
out_dir = KML_OUT_DIR_LARUS
files = sorted(glob.glob(f"{input_dir}/*.parquet"))

if not files:
    logger.error("No parquet files found in %s", input_dir)
else:
    print(f"Found {len(files)} files.")
    results, f37_thermals_all = process_all(files, out_dir)

    # ✅ ensure filename column exists
    if "Filename" not in f37_thermals_all.columns:
        f37_thermals_all["Filename"] = "Unknown"

    if not f37_thermals_all.empty:
        os.makedirs("Results/dataframes", exist_ok=True)
        f37_thermals_all.to_parquet("Results/dataframes/f37_thermals_all.parquet", index=False)
        print("\nIndex → filename:")
        for i, (fname, *_ ) in enumerate(results):
            print(f"{i} - {fname}")

        rows = []
        for fname, grp in f37_thermals_all.groupby("Filename"):
            n_thermals = grp["ThermalNumber"].nunique()
            rows.append({"Filename": fname, "Thermals detected": n_thermals})
        summary_df = pd.DataFrame(rows)
        print("\n=== Summary Table ===")
        print(summary_df.to_string(index=False))
    else:
        logger.warning("No thermals detected in any file.")




In [ ]:
# Objective 3.2 :  detect thermal regions, interpolate wind info and build igc_thermals_all.parquet for IGC flights
# 1. Detect thermal regions using progress < 0.9 & climbRate > 0
# 2. Filter out those with alt gain < 50 m
# 3. interpolate wind from LARUS data and thereby calculate TE rate
# 4. Save thermal dataframe, KML files and print summary table

if EXPORT_KML:
    os.makedirs(KML_OUT_DIR_IGC, exist_ok=True)

# ---------- Helpers ----------
def load_larus_for_date(candidates=LARUS_CANDIDATES, date_str=SELECTED_DATE):
    y, m, d = map(int, date_str.split("-"))
    lar = None
    for fp in candidates:
        if os.path.exists(fp):
            lar = pd.read_parquet(fp); break
    if lar is None: raise FileNotFoundError("No LARUS parquet found among candidates.")

    lar.columns = [c.strip().replace("  ", " ") for c in lar.columns]
    lar[["year","month","day"]] = lar[["year","month","day"]].astype(int)
    if lar["year"].max() < 100: lar["year"] += 2000
    lar = lar[(lar["year"]==y)&(lar["month"]==m)&(lar["day"]==d)].copy()
    if lar.empty: raise ValueError(f"No LARUS rows for {date_str}.")

    for wn,we in [("inst wind corrected N","inst wind corrected E"),
                  ("inst wind N","inst wind E"),
                  ("wind N","wind E"),
                  ("wind avg N","wind avg E")]:
        if wn in lar.columns and we in lar.columns:
            lar = lar.rename(columns={wn:"wind_N", we:"wind_E"}); break
    else:
        raise ValueError("No usable wind component columns (N/E).")

    ts = pd.to_datetime(dict(year=lar["year"], month=lar["month"], day=lar["day"],
                             hour=lar["hour"].astype(int), minute=lar["minute"].astype(int),
                             second=lar["second"].astype(int)), utc=True)
    if "nanoseconds" in lar.columns and lar["nanoseconds"].notna().any():
        ts = ts + pd.to_timedelta(lar["nanoseconds"].fillna(0).astype(int), unit="ns")
    lar["timestamp"] = ts.view("int64")/1e9

    lar = (lar.dropna(subset=["Lat","Long","wind_N","wind_E","timestamp"])
              .rename(columns={"Lat":"lat","Long":"lon"})
              .sort_values("timestamp"))

    # downsample to 1 Hz to speed up temporal join
    lar["t"] = pd.to_datetime(lar["timestamp"], unit="s").dt.floor("1s")
    lar = (lar.groupby("t", as_index=False)
             .agg(lat=("lat","mean"), lon=("lon","mean"),
                  wind_N=("wind_N","mean"), wind_E=("wind_E","mean")))
    lar["timestamp"] = lar["t"].view("int64")/1e9
    return lar[["t","timestamp","lat","lon","wind_N","wind_E"]].reset_index(drop=True)

def _haversine_km_vec(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    dlat, dlon = lat2-lat1, lon2-lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 6371.0088 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

def attach_wind_components_fast(df, lar, t_half=T_HALF, r_km=R_KM):
    """Return df with wind_N, wind_E columns (no wind_speed/dir)."""
    igc = df[["timestamp","lat","lon"]].rename(columns={"timestamp":"t_igc"}).copy()
    igc["t"] = pd.to_datetime(igc["t_igc"], unit="s")
    lar = lar.sort_values("t")

    medN = float(np.nanmedian(lar["wind_N"])); medE = float(np.nanmedian(lar["wind_E"]))
    tol  = pd.Timedelta(seconds=2*t_half)

    back = pd.merge_asof(igc.sort_values("t"), lar, on="t", direction="backward", tolerance=tol,
                         suffixes=("_igc","_larus"))
    fwd  = pd.merge_asof(igc.sort_values("t"), lar, on="t", direction="forward",  tolerance=tol,
                         suffixes=("_igc","_larus"))

    wn = np.full(len(igc), medN, dtype=float)
    we = np.full(len(igc), medE, dtype=float)

    b_ok = back["wind_N"].notna().to_numpy()
    f_ok = fwd ["wind_N"].notna().to_numpy()

    if b_ok.any():
        wn[b_ok] = back.loc[b_ok,"wind_N"].to_numpy()
        we[b_ok] = back.loc[b_ok,"wind_E"].to_numpy()

    if f_ok.any():
        # where both exist, blend by time+distance weights; else use whichever exists
        dist_b = _haversine_km_vec(igc.loc[b_ok,"lat"], igc.loc[b_ok,"lon"],
                                   back.loc[b_ok,"lat_larus"], back.loc[b_ok,"lon_larus"]) if b_ok.any() else np.array([])
        dist_f = _haversine_km_vec(igc.loc[f_ok,"lat"], igc.loc[f_ok,"lon"],
                                   fwd.loc[f_ok,"lat_larus"], fwd.loc[f_ok,"lon_larus"])
        dt_b = np.abs(igc.loc[b_ok,"t_igc"].to_numpy() - back.loc[b_ok,"t"].view("int64")/1e9) if b_ok.any() else np.array([])
        dt_f = np.abs(igc.loc[f_ok,"t_igc"].to_numpy() - fwd.loc[f_ok,"t"].view("int64")/1e9)
        w_b = 1/(1+(dist_b/r_km)**2) * 1/(1+(dt_b/t_half)**2) if b_ok.any() else np.array([])
        w_f = 1/(1+(dist_f/r_km)**2) * 1/(1+(dt_f/t_half)**2)

        both = np.zeros(len(igc), dtype=bool); both[np.where(b_ok)[0]] = True; both &= f_ok
        only_f = f_ok & ~b_ok

        if both.any():
            # indices in original arrays
            idx_both = np.where(both)[0]
            wb = w_b[np.isin(np.where(b_ok)[0], idx_both)]
            wf = w_f[np.isin(np.where(f_ok)[0], idx_both)]
            wn[both] = (wb*back.loc[both,"wind_N"].to_numpy() + wf*fwd.loc[both,"wind_N"].to_numpy())/(wb+wf+1e-9)
            we[both] = (wb*back.loc[both,"wind_E"].to_numpy() + wf*fwd.loc[both,"wind_E"].to_numpy())/(wb+wf+1e-9)

        if only_f.any():
            wn[only_f] = fwd.loc[only_f,"wind_N"].to_numpy()
            we[only_f] = fwd.loc[only_f,"wind_E"].to_numpy()

    out = df.copy()
    out["wind_N"] = wn; out["wind_E"] = we
    return out

def haversine_km(lat1, lon1, lat2, lon2):
    R=6371.0088; φ1,φ2=math.radians(lat1),math.radians(lat2)
    Δφ,Δλ=math.radians(lat2-lat1),math.radians(lon2-lon1)
    a=math.sin(Δφ/2)**2+math.cos(φ1)*math.cos(φ2)*math.sin(Δλ/2)**2
    return R*2*math.atan2(math.sqrt(a),math.sqrt(1-a))

def flight_to_df(f):
    t0 = f.fixes[0].timestamp
    return pd.DataFrame([{
        "timestamp": float(x.timestamp),
        "flighttime": float(x.timestamp)-float(t0),
        "lat": float(x.lat), "lon": float(x.lon), "alt": float(x.gnss_alt),
        "gsp": float(x.gsp)/3.6,                    # km/h -> m/s
        "bearing": (float(x.bearing)+360.0)%360.0,  # -180..180 -> 0..360
        "bearing_change_rate": float(x.bearing_change_rate)
    } for x in f.fixes])

def classify_thermals(df, fname="", window_sec=20, thresh=0.9, min_segment_sec=10, min_dur=30, min_gain=50):
    n=len(df)
    if n<3: return pd.DataFrame(),[],pd.DataFrame()
    seg_dist=[haversine_km(df.lat[i],df.lon[i],df.lat[i+1],df.lon[i+1]) for i in range(n-1)]
    i=j=0; out=[]; thermals=[]; segments=[]; tc=1
    pbar=tqdm(total=n-1, desc=f"Classifying {os.path.basename(fname)}", unit="steps")
    while i<n-1:
        while j<n-1 and (df.flighttime[j]-df.flighttime[i])<window_sec:j+=1
        if j==i: i+=1; pbar.update(1); continue
        track=haversine_km(df.lat[i],df.lon[i],df.lat[j],df.lon[j])
        ground=sum(seg_dist[i:j]) if j>i else 0; progress=track/ground if ground>0 else 1.0
        dt=df.flighttime[j]-df.flighttime[i]; cr=(df.alt[j]-df.alt[i])/dt if dt>0 else 0
        label="Glide" if progress>thresh else ("Thermal" if cr>0 else "EmergencyDescent")
        out.append({"idx":i,"t":df.flighttime[i],"progress":progress,"climbRate":cr,"label":label}); i+=1; pbar.update(1)
    pbar.close(); df_cls=pd.DataFrame(out)

    in_th=False; si=None
    for k in range(len(df_cls)):
        is_th=df_cls.label.iat[k]=="Thermal"
        if is_th and not in_th: in_th=True; si=int(df_cls.idx.iat[k])
        elif not is_th and in_th:
            in_th=False; ei=int(df_cls.idx.iat[k]); dur=df.flighttime[ei]-df.flighttime[si]; gain=df.alt[ei]-df.alt[si]
            if dur>=min_dur and gain>=min_gain and (ei-si)>=min_segment_sec:
                sub=df.iloc[si:ei+1].copy()
                sub["time"]=pd.to_datetime(sub["timestamp"],unit="s").dt.strftime("%H:%M:%S")
                sub["ThermalNumber"]=tc
                thermals.append(sub[["ThermalNumber","flighttime","timestamp","time","lat","lon","alt","gsp","bearing","bearing_change_rate"]])
                segments.append(dict(start_idx=si,end_idx=ei,start_t=df.flighttime[si],end_t=df.flighttime[ei])); tc+=1
    if in_th:
        ei=n-1; dur=df.flighttime[ei]-df.flighttime[si]; gain=df.alt[ei]-df.alt[si]
        if dur>=min_dur and gain>=min_gain and (ei-si)>=min_segment_sec:
            sub=df.iloc[si:ei+1].copy()
            sub["time"]=pd.to_datetime(sub["timestamp"],unit="s").dt.strftime("%H:%M:%S")
            sub["ThermalNumber"]=tc
            thermals.append(sub[["ThermalNumber","flighttime","timestamp","time","lat","lon","alt","gsp","bearing","bearing_change_rate"]])
            segments.append(dict(start_idx=si,end_idx=ei,start_t=df.flighttime[si],end_t=df.flighttime[ei]))
    return df_cls,segments,(pd.concat(thermals,ignore_index=True) if thermals else pd.DataFrame())

def split_all_flights(igc_parsed, igc_files, ws=20, th=0.9, ms=10):
    results=[]
    for f,fn in zip(igc_parsed,igc_files):
        db=flight_to_df(f)
        dc,segs,ith=classify_thermals(db, fname=fn, window_sec=ws, thresh=th, min_segment_sec=ms)
        ith["Filename"]=fn
        results.append((fn,db,dc,segs,ith))
    return results, pd.concat([r[4] for r in results], ignore_index=True)

def export_thermals_to_kml(df_base, thermals, out_path, label="Thermal"):
    kml = Kml()
    flight_coords = list(zip(df_base.lon, df_base.lat, df_base.alt))
    ls = kml.newlinestring(name="Full Flight Path", coords=flight_coords)
    ls.style.linestyle.color = Color.gray; ls.style.linestyle.width = 2
    ls.altitudemode = AltitudeMode.absolute
    for th_num, th in thermals.groupby("ThermalNumber"):
        coords = list(zip(th.lon, th.lat, th.alt))
        ls = kml.newlinestring(name=f"{label} {th_num}", coords=coords)
        ls.style.linestyle.width = 3
        ls.style.linestyle.color = Color.rgb(random.randint(0,255), random.randint(0,255), random.randint(0,255), a=200)
        ls.altitudemode = AltitudeMode.absolute
        dur = th.flighttime.iloc[-1]-th.flighttime.iloc[0]
        alt_gain = th.alt.iloc[-1]-th.alt.iloc[0]
        ls.description = f"Duration: {dur:.1f}s<br>Δalt: {alt_gain:.1f} m"
    kml.save(out_path)

# ---------------- MAIN ----------------
igc_files  = [f for f in igc_files  if SELECTED_DATE in os.path.basename(f)]
igc_parsed = [p for f,p in zip(igc_files, igc_parsed) if SELECTED_DATE in os.path.basename(f)]

results, igc_thermals_all = split_all_flights(igc_parsed, igc_files)
larus_df = load_larus_for_date()

# Attach wind from LARUS (per flight), then concat
aug = []
for fn, db, _, _, ith in results:
    db_w  = attach_wind_components_fast(db, larus_df, t_half=T_HALF, r_km=R_KM)
    ith_w = ith.merge(db_w[["timestamp","lat","lon","wind_N","wind_E"]],
                      on=["timestamp","lat","lon"], how="left")
    ith_w["Filename"] = fn
    aug.append(ith_w)
igc_thermals_all = pd.concat(aug, ignore_index=True)

per_flight_te = {}
for fn, db, _, _, ith in results:
    db_w = attach_wind_components_fast(db, larus_df, t_half=T_HALF, r_km=R_KM)

    brad = np.deg2rad(db_w["bearing"].to_numpy())
    G_N  = db_w["gsp"].to_numpy()*np.cos(brad)
    G_E  = db_w["gsp"].to_numpy()*np.sin(brad)
    TAS  = np.hypot(G_N - db_w["wind_N"].to_numpy(), G_E - db_w["wind_E"].to_numpy())

    db_w["TotalEnergy_full"] = g*db_w["alt"].to_numpy() + 0.5*TAS**2
    per_flight_te[fn] = db_w[["timestamp","flighttime","TotalEnergy_full","lat","lon"]]


# TAS (from ground vector minus wind)
brad = np.deg2rad(pd.to_numeric(igc_thermals_all["bearing"], errors="coerce"))
G_N  = igc_thermals_all["gsp"]*np.cos(brad)
G_E  = igc_thermals_all["gsp"]*np.sin(brad)
W_N  = pd.to_numeric(igc_thermals_all["wind_N"], errors="coerce")
W_E  = pd.to_numeric(igc_thermals_all["wind_E"], errors="coerce")
igc_thermals_all["TAS"] = np.hypot(G_N - W_N, G_E - W_E)

# --- Build filtered/smoothed fields only (no temp cols kept) ---
# jitter to ensure strictly increasing time in groups
igc_thermals_all["flighttime2"] = (
    pd.to_numeric(igc_thermals_all["flighttime"], errors="coerce")
    + igc_thermals_all.groupby(["Filename","ThermalNumber","flighttime"]).cumcount()*1e-2
)

# altitude smoothing (interpolate -> median -> mean)
alt_interp = (
    igc_thermals_all.groupby(["Filename","ThermalNumber"])["alt"]
    .transform(lambda s: pd.to_numeric(s, errors="coerce").interpolate("linear", limit_direction="both"))
)
ALT_filt = (alt_interp.rolling(WIN, center=True, min_periods=1).median()
                     .rolling(WIN, center=True, min_periods=1).mean())

# TAS smoothing (median -> mean)
TAS_filt = (pd.to_numeric(igc_thermals_all["TAS"], errors="coerce")
            .rolling(WIN, center=True, min_periods=1).median()
            .rolling(WIN, center=True, min_periods=1).mean())

igc_thermals_all["ALT_filt"] = ALT_filt
igc_thermals_all["TAS_filt"] = TAS_filt

# sort + reset index so later groupwise outputs align by position
igc_thermals_all = igc_thermals_all.sort_values(["Filename","ThermalNumber","flighttime2"]).reset_index(drop=True)

def _grad_series(y, t):
    yv = pd.to_numeric(y, errors="coerce").to_numpy()
    tv = pd.to_numeric(t, errors="coerce").to_numpy()
    if len(yv) < 2 or np.all(~np.isfinite(tv)):
        return pd.Series(np.full(len(yv), np.nan))
    yclean = pd.Series(yv).interpolate("linear", limit_direction="both").to_numpy()
    return pd.Series(np.gradient(yclean, tv))

# derivatives (aligned by position due to reset_index above)
dALT = (igc_thermals_all.groupby(["Filename","ThermalNumber"], group_keys=False)
        .apply(lambda g: _grad_series(g["ALT_filt"], g["flighttime2"])).reset_index(drop=True))
dTAS = (igc_thermals_all.groupby(["Filename","ThermalNumber"], group_keys=False)
        .apply(lambda g: _grad_series(g["TAS_filt"], g["flighttime2"])).reset_index(drop=True))
igc_thermals_all["dALT_dt"] = dALT
igc_thermals_all["dTAS_dt"] = dTAS

# TE rate (raw) and smoothed (instrument-like)
TE_rate_raw = g*igc_thermals_all["dALT_dt"] + igc_thermals_all["TAS_filt"]*igc_thermals_all["dTAS_dt"]

TE_clip = TE_rate_raw.clip(-200, 300)

def _iir_smooth(y, t, tau=1.8):
    y = pd.to_numeric(y, errors="coerce").to_numpy()
    t = pd.to_numeric(t, errors="coerce").to_numpy()
    out = np.full(len(y), np.nan, dtype=float)
    if len(y)==0: return out
    out[0] = 0 if not np.isfinite(y[0]) else y[0]
    for k in range(1, len(y)):
        dt = t[k]-t[k-1]
        a  = 0 if not np.isfinite(dt) or dt<=0 else (1 - np.exp(-dt/tau))
        val= y[k] if np.isfinite(y[k]) else out[k-1]
        out[k] = out[k-1] + a*(val - out[k-1])
    return out

def _iir_smooth_series(g):
    out = _iir_smooth(TE_clip.loc[g.index], g["flighttime2"])
    # return a Series with the SAME index -> pandas keeps float dtype
    return pd.Series(out, index=g.index, dtype="float64")

igc_thermals_all["TotalEnergy_rate"] = (
    igc_thermals_all
    .groupby(["Filename","ThermalNumber"], group_keys=False)
    .apply(_iir_smooth_series)
)
igc_thermals_all["TotalEnergy"] = g*igc_thermals_all["alt"] + 0.5*igc_thermals_all["TAS"]**2

# ---- Final compact dataframe ----
igc_thermals_all["label"]    = "Thermal"
igc_thermals_all["filetype"] = "igc"

final_df = pd.DataFrame({
    "flighttime":          igc_thermals_all["flighttime"],
    "lat":                 igc_thermals_all["lat"],
    "lon":                 igc_thermals_all["lon"],
    "alt":                 igc_thermals_all["alt"],
    "TAS":                 igc_thermals_all["TAS_filt"],
    "bearing":             igc_thermals_all["bearing"],
    "wind_N":              igc_thermals_all["wind_N"],
    "wind_E":              igc_thermals_all["wind_E"],
    "time":                igc_thermals_all["time"],
    "timestamp":           igc_thermals_all["timestamp"],
    "TotalEnergy":         igc_thermals_all["TotalEnergy"],
    "TotalEnergy_rate":    igc_thermals_all["TotalEnergy_rate"],
    "label":               igc_thermals_all["label"],
    "ThermalNumber":       igc_thermals_all["ThermalNumber"],
    "Filename":            igc_thermals_all["Filename"],
    "gsp":                 igc_thermals_all["gsp"],
    "turn_rate": igc_thermals_all["bearing_change_rate"],
    "filetype":            igc_thermals_all["filetype"],
})

final_df.to_parquet("Results/dataframes/igc_thermals_all.parquet", index=False)

# --- Optional: quick summary & KML (you can delete if not needed) ---
rows = [{"Filename": fn, "Thermals (detected)": grp["ThermalNumber"].nunique()}
        for fn, grp in final_df.groupby("Filename")]
print("\n=== Summary Table ===")
print(pd.DataFrame(rows).to_string(index=False))

for fn, db, _, _, _ in results:
    before = final_df[final_df["Filename"]==fn]
    if EXPORT_KML and not before.empty:
        out_before = os.path.join(KML_OUT_DIR_IGC, os.path.basename(fn).replace(".igc","_thermals.kml"))
        export_thermals_to_kml(db, before, out_before, label="Thermal")


## 4. Load Task and Thermal Dataframe

In [ ]:
# Objective 4.1 : load task.json and plot together with thermals on map, cluster thermal centers, and calculate stats
# 1. load task from json
# 2. load thermals from f37 parquet + igc files
# 3. cluster thermal centers (DBSCAN, Haversine) range = 0.5
# 4. plot task and thermals in one map using plotly
# 5. cluster thermal centers
# 6. calculate number of pilots and mean climb rate per cluster

# --- load task ---
with open("Raw_data/task/2025-06-09.json", "r") as f:
    data = json.load(f)
points = data["point_features"]
df_task = pd.DataFrame([{
    "name": p["properties"]["name"],
    "elevation": p["properties"]["elevation"],
    "lon": p["geometry"]["coordinates"][0],
    "lat": p["geometry"]["coordinates"][1]
} for p in points])

# --- load thermals (f37 + igc) ---
f37 = pd.read_parquet("Results/dataframes/f37_thermals_all.parquet")
igc = pd.read_parquet("Results/dataframes/igc_thermals_all.parquet")

thermals = pd.concat([f37, igc], ignore_index=True)
thermals_all = thermals[thermals["Filename"].str.contains("2025-06-09")].copy()

thermals_all['filetype'] = np.where(
    thermals_all['Filename'].str.endswith('.parquet'), 'f37',
    np.where(thermals_all['Filename'].str.endswith('.igc'), 'igc', 'unknown')
)

# Downsample:
def downsample_group(df):
    if ".parquet" in df["Filename"].iloc[0]:
        return df.iloc[::10]
    else:
        return df

# Left join thermal_centers into thermals_ds on Filename and ThermalNumber
thermals_ds = (
    thermals_all
    .groupby(["Filename", "ThermalNumber"], group_keys=False)
    .apply(downsample_group)
    .reset_index(drop=True)
)

thermal_centers = (
    thermals_all.groupby(["Filename","ThermalNumber"])
    .agg({
        "lat":"mean",
        "lon":"mean",
        "flighttime":"median"
    })
    .reset_index()
)

# --- Convert lat/lon to radians ---
coords = np.radians(thermal_centers[['lat','lon']].values)

# --- Define merge distance threshold (in km) ---
eps = R_MERGE_KM / R_EARTH_KM

# --- Cluster thermal centers using DBSCAN (Haversine) ---
clustering = DBSCAN(eps=eps, min_samples=1, metric='haversine').fit(coords)
thermal_centers['thermal_group'] = clustering.labels_
n_clusters = len(set(thermal_centers['thermal_group'])) - (1 if -1 in clustering.labels_ else 0)
print(f"Detected {n_clusters} clusters (r_merge = {R_MERGE_KM} km)")

thermals_ds = thermals_ds.merge(
    thermal_centers[["Filename", "ThermalNumber", "thermal_group"]],
    on=["Filename", "ThermalNumber"],
    how="left"
)

thermals_ds["alt"] = pd.to_numeric(thermals_ds["alt"], errors="coerce")

def trim_alt_outliers_per_group(df):
    def _trim(g):
        a = g["alt"].dropna()
        if len(a) < 10:
            return g
        m  = a.mean()
        s  = a.std()
        if not np.isfinite(s) or s == 0:
            return g
        #upper = m + 3 * s
        lower = m - 4 * s  # your rule
        return g[(g["alt"] >= lower)] #g["alt"] <= upper) & (
    return (df.groupby("thermal_group", group_keys=False)
              .apply(_trim)
              .reset_index(drop=True))

thermals_ds = trim_alt_outliers_per_group(thermals_ds)

t_strict = pd.to_datetime(thermals_ds['time'], format='%H:%M:%S', errors='coerce')
mask = t_strict.isna()
if mask.any():
    t_fallback = pd.to_datetime(thermals_ds.loc[mask, 'time'], errors='coerce')
    t_strict.loc[mask] = t_fallback
thermals_ds['time_dt'] = t_strict

thermals_ds_path = "Results/dataframes/thermals_all_reduced.parquet"
thermals_ds.to_parquet(thermals_ds_path, index=False)
print(f"✅ Saved downsampled thermals -> {thermals_ds_path}")

# --- Build a discrete color map for thermal_group ---
unique_groups = sorted(thermal_centers["thermal_group"].unique())
base_colors = px.colors.qualitative.Set2 + px.colors.qualitative.Bold + px.colors.qualitative.Pastel
color_discrete_map = {str(g): base_colors[i % len(base_colors)] for i, g in enumerate(unique_groups)}

# --- Base map: task line + waypoints ---
fig = px.scatter_map(
    df_task,
    lat="lat", lon="lon",
    hover_name="name",
    hover_data={"elevation": True, "lat": False, "lon": False},
    map_style="open-street-map",
    zoom=6, height=650
)
fig.update_traces(mode="lines+markers", showlegend=False)

# --- Thermals (colored by thermal_group) ---
fig_th = px.scatter_map(
    thermals_ds,
    lat="lat", lon="lon",
    color=thermals_ds["thermal_group"].astype(str),
    hover_data={
        "Filename": True,
        "flighttime": True,
        "ThermalNumber": True,
        "thermal_group": True
    },
    opacity=0.6,
    color_discrete_map=color_discrete_map,
    title=f"Task and Thermals with Clustered Centers<br>({n_clusters} clusters detected)"
)
fig_th.update_traces(marker=dict(size=5), showlegend=True)

for tr in fig_th.data:
    fig.add_trace(tr)

# --- Thermal centers (cluster markers, same color) ---
fig_centers = px.scatter_map(
    thermal_centers,
    lat="lat", lon="lon",
    color=thermal_centers["thermal_group"].astype(str),
    hover_data={
        "Filename": True,
        "ThermalNumber": True,
        "thermal_group": True
    },
    opacity=0.9,
    color_discrete_map=color_discrete_map
)
fig_centers.update_traces(marker=dict(size=10, symbol="x"), showlegend=True)

for tr in fig_centers.data:
    fig.add_trace(tr)

# --- Layout ---
fig.update_layout(
    mapbox_style="open-street-map",
    legend_title="Thermal Group",
    title="Task and Thermals Colored by Cluster (thermal_group)",
    height=700
)


# Add thermal_group label
thermals_all = thermals_all.merge(
    thermal_centers[["Filename", "ThermalNumber", "thermal_group"]],
    on=["Filename", "ThermalNumber"],
    how="left"
)

parquet_path = "Results/dataframes/thermals_all.parquet"
thermals_all.to_parquet(parquet_path, index=False)
print(f"✅ Saved thermals all → {parquet_path}")


#fig.show()
'''os.makedirs("Results/plots", exist_ok=True)
fig.write_html("Results/plots/thermals_clustered_map.html", auto_open=False)
print("✅ Saved to Results/plots/thermals_clustered_map.html")'''




In [ ]:
# Objective 4.2 : Clustering + cluster table + strength-class means
# 1. Classification of total energy gain per thermal cluster into strength classes (Weak, Moderate, Good, Strong)
# 2. Summarization of thermal clusters into a cluster table

def utm_xy(lat: np.ndarray, lon: np.ndarray, lat0: float, lon0: float):
    """Project lat/lon to UTM meters centered on (lat0, lon0)."""
    zone = int((lon0 + 180) / 6) + 1
    proj = pyproj.Proj(proj="utm", zone=zone, ellps="WGS84", always_xy=True)
    x_raw, y_raw = proj(lon, lat)
    x0, y0 = proj(lon0, lat0)
    return x_raw - x0, y_raw - y0, {"utm_zone": zone, "x_center": float(x0), "y_center": float(y0)}

# --------- UTM coordinates in meters (common zone) ----------
lat0 = thermals_ds["lat"].mean()
lon0 = thermals_ds["lon"].mean()

thermals_ds["x_m"], thermals_ds["y_m"], utm_meta = utm_xy(
    thermals_ds["lat"].to_numpy(), thermals_ds["lon"].to_numpy(), lat0, lon0
)
# -------------------- Cluster strength table (by TotalEnergy GAIN per thermal) --------------------
# Compute TE for each (Filename, ThermalNumber) inside each thermal_group
thermals_sorted = thermals_ds.sort_values(
    ["Filename", "ThermalNumber", "flighttime"]
)

# ?TE per (thermal_group, Filename, ThermalNumber) using iloc
per_thermal = (
    thermals_sorted
    .groupby(["thermal_group", "Filename", "ThermalNumber"])["TotalEnergy"]
    .agg(lambda s: s.iloc[-1] - s.iloc[0])
    .reset_index(name="TE_gain")
)

# Aggregate to cluster level (energy-based stats)
cluster_strength = (
    per_thermal
    .groupby("thermal_group", dropna=False)
    .agg(
        te_gain_mean=("TE_gain", "mean"),
        te_gain_sum =("TE_gain", "sum"),
        n_thermals =("TE_gain", "size"),
        n_pilots   =("Filename", "nunique"),
    )
    .reset_index()
)

# Optimum (mean) flight parameters per thermal_group
# turn_rate_mean is based on absolute turn_rate
opt_flight_params = (
    thermals_ds
    .assign(turn_rate_abs=lambda df: df["turn_rate"].abs())
    .groupby("thermal_group", dropna=False)
    .agg(
        TAS_mean       = ("TAS", "mean"),          # [m/s]
        turn_rate_mean = ("turn_rate_abs", "mean") # [deg/s], mean(|turn_rate|)
    )
    .reset_index()
)

# Merge flight parameters into cluster_strength
cluster_strength = cluster_strength.merge(
    opt_flight_params,
    on="thermal_group",
    how="left"
)

# Flag clusters that have at least one f37 segment
has_f37 = (
    thermals_ds.groupby("thermal_group")["filetype"]
    .apply(lambda s: int((s == "f37").any()))
    .reset_index(name="has_f37")
)

cluster_strength = cluster_strength.merge(has_f37, on="thermal_group", how="left")

# Class (weak→strong) by te_gain_mean quantiles
if not cluster_strength.empty and cluster_strength["te_gain_mean"].notna().any():
    qs = cluster_strength["te_gain_mean"].quantile([0.25,0.5,0.75]).values
    def _class(v):
        if np.isnan(v): return "Unknown"
        if v <= qs[0]: return "Weak"
        elif v <= qs[1]: return "Moderate"
        elif v <= qs[2]: return "Good"
        else: return "Strong"
    cluster_strength["strength_class"] = cluster_strength["te_gain_mean"].apply(_class)
else:
    cluster_strength["strength_class"] = "Unknown"

# Score to pick the best group (clip negatives so down-energy thermals don't help)
cluster_strength["score_mult"] = (
    cluster_strength["n_thermals"]
    * cluster_strength["te_gain_mean"].clip(lower=0).fillna(0)
    * cluster_strength["has_f37"].fillna(0)
)

print("\n=== Thermal Cluster Table (by TotalEnergy GAIN per thermal) ===")

# sort by: 
#   1) n_thermals   (descending)
#   2) te_gain_mean (descending)
cluster_strength_sorted = (
    cluster_strength
    .sort_values(["n_thermals", "te_gain_mean"], ascending=[False, False])
)

print(
    cluster_strength_sorted[
        [
            "thermal_group",
            "n_pilots",
            "n_thermals",
            "te_gain_mean",
            "TAS_mean",
            "turn_rate_mean",
            "strength_class",
            "has_f37",
        ]
    ].to_string(index=False)
)

# Mean TAS and |turn rate| per strength class
strength_params = (
    cluster_strength
    .groupby("strength_class", dropna=False)
    .agg(
        TAS_mean       = ("TAS_mean", "mean"),
        turn_rate_mean = ("turn_rate_mean", "mean"),
    )
    .reset_index()
    .sort_values("strength_class")
)

strength_params


In [ ]:
# Objective 4.3 : Select top group and build map + Dash app + configure modelling

# --- high-level switches ---

RUN_MODELLING_LOOPS = 1      # 0: only selected thermal group; 1: loop over top-N groups
top_group = 25               # used when RUN_MODELLING_LOOPS == 0
# top_group = int(cluster_strength_sorted.iloc[0]["thermal_group"])
N_TOP_GROUPS = 100            # <--- NEW: set 10, 50, etc; None => all ranked

# per-plot switches (0 = skip, 1 = generate)
PLOT_DRIFT_TE_RATE                = 0
PLOT_XY_SLICES_TERATE             = 0
PLOT_MEAN_PLANE_XY_TERATE         = 0
PLOT_ROLLING_TE_PROFILES          = 0
PLOT_THERMAL_CYLINDER             = 0
PLOT_ANIMATED_CYLINDER_PATHS      = 0
PLOT_ANIMATED_CYLINDER_DRIFT_FREE = 0
PLOT_MODEL_COMPARISON             = 0
RUN_DASH_APP                      = 0
COMPUTE_THERMALLING_EFFICIENCY    = 1

VERBOSE = False          # set True if you want detailed logging
PLOT_LOG: list[str] = [] # will collect plot file paths per thermal group

# automatically disable Dash when looping over groups
if RUN_MODELLING_LOOPS:
    if N_TOP_GROUPS is not None:
        modelling_groups = (
            cluster_strength_sorted["thermal_group"]
            .dropna()
            .astype(int)
            .head(N_TOP_GROUPS)
            .tolist()
        )
    else:
        modelling_groups = (
            cluster_strength_sorted["thermal_group"]
            .dropna()
            .astype(int)
            .tolist()
        )
else:
    modelling_groups = [int(top_group)]

print("\n=== Modelling config ===")
print(f"RUN_MODELLING_LOOPS = {RUN_MODELLING_LOOPS}")
print(f"top_group (single-group mode) = {top_group}")
print(f"modelling_groups = {modelling_groups}")

print("\n=== Building map and 3D Dash app ===")


# -------------------- Map (task + thermals + centers) --------------------
unique_groups = sorted(thermal_centers["thermal_group"].unique())
base_colors = px.colors.qualitative.Set2 + px.colors.qualitative.Bold + px.colors.qualitative.Pastel
color_discrete_map = {str(g): base_colors[i % len(base_colors)] for i, g in enumerate(unique_groups)}

fig = px.scatter_map(
    df_task, lat="lat", lon="lon",
    hover_name="name",
    hover_data={"elevation":True,"lat":False,"lon":False},
    map_style="open-street-map", zoom=6, height=650
)
fig.update_traces(mode="lines+markers", showlegend=False)

thermals_ds = (
    thermals_ds.groupby(["Filename","ThermalNumber"], group_keys=False)
    .apply(downsample_group).reset_index(drop=True)
)

fig_th = px.scatter_map(
    thermals_ds, lat="lat", lon="lon",
    color=thermals_ds["thermal_group"].astype(str),
    hover_data={"Filename":True,"ThermalNumber":True,"thermal_group":True},
    opacity=0.6, color_discrete_map=color_discrete_map,
    title="Task and Thermals with Clustered Centers"
)
fig_th.update_traces(marker=dict(size=5), showlegend=True)
for tr in fig_th.data:
    fig.add_trace(tr)

fig_ctr = px.scatter_map(
    thermal_centers, lat="lat", lon="lon",
    color=thermal_centers["thermal_group"].astype(str),
    hover_data={"Filename":True,"ThermalNumber":True,"thermal_group":True},
    opacity=0.9, color_discrete_map=color_discrete_map
)
fig_ctr.update_traces(marker=dict(size=10, symbol="x"), showlegend=True)
for tr in fig_ctr.data:
    fig.add_trace(tr)

fig.update_layout(
    legend_title="Thermal Group",
    title="Task and Thermals Colored by Cluster (thermal_group)",
    height=700
)
os.makedirs("Results/plots", exist_ok=True)
fig.write_html("Results/plots/thermals_clustered_map.html", auto_open=False)
print("✅ Saved → Results/plots/thermals_clustered_map.html")

# -------------------- Dash 3D (color by prebuilt TotalEnergy_rate) --------------------
#top_group = int(cluster_strength.loc[cluster_strength["score_mult"].idxmax(), "thermal_group"])

#print(f"Best overall thermal group = {top_group}")

if not RUN_MODELLING_LOOPS and RUN_DASH_APP:
    print(f"Best overall thermal group = {top_group}")

    df_top = (
        thermals_ds[thermals_ds["thermal_group"]==top_group]
        .sort_values(["Filename","ThermalNumber","flighttime"])
        .copy()
    )

    # center coordinates for plotting (per top_group)
    x0_plot = df_top["x_m"].mean()
    y0_plot = df_top["y_m"].mean()

    # precompute centered versions for convenience
    df_top["x_plot"] = df_top["x_m"] - x0_plot
    df_top["y_plot"] = df_top["y_m"] - y0_plot

    # symmetric extent for x/y so circles are not visually stretched
    r_xy = max(df_top["x_plot"].abs().max(), df_top["y_plot"].abs().max())
    X_MIN, X_MAX = -r_xy, r_xy
    Y_MIN, Y_MAX = -r_xy, r_xy

    # Optional: harmonize f37 sampling to IGC
    f37_data = df_top[df_top["filetype"]=="f37"]
    igc_data = df_top[df_top["filetype"]=="igc"]
    if not f37_data.empty and not igc_data.empty:
        f37_dt = f37_data.groupby("Filename")["flighttime"].diff().median()
        igc_dt = igc_data.groupby("Filename")["flighttime"].diff().median()
        if pd.notna(f37_dt) and pd.notna(igc_dt) and f37_dt>0:
            ratio = max(1, int(round(igc_dt / f37_dt)))
            parts = []
            for name, gdf in df_top.groupby("Filename"):
                if gdf["filetype"].iloc[0] == "f37":
                    gdf = gdf.iloc[::ratio].copy()
                parts.append(gdf)
            df_top = pd.concat(parts, ignore_index=True).sort_values(["Filename","ThermalNumber","flighttime"])

    filenames = sorted(df_top["Filename"].unique())
    color_field = "TotalEnergy_rate"

    # robust global color range
    q1, q99 = -10, 50
    cmin, cmax = float(q1), float(q99)

    app = Dash(__name__)
    app.title = f"Thermal Group {int(top_group)} — 3D Visualization"

    app.layout = html.Div([
        html.H2(f"3D Flight Paths — Thermal Group {int(top_group)}"),
        html.P("Select pilots to display:"),
        html.Div([
            dcc.Dropdown(
                id="pilot-dropdown",
                options=[{"label": os.path.basename(n), "value": n} for n in filenames],
                value=filenames[:5],
                multi=True, placeholder="Select pilots...", style={"width":"80%"}
            ),
            html.Button("Select All", id="select-all-btn", n_clicks=0, style={"marginLeft":"12px"}),
        ], style={"display":"flex","alignItems":"center","marginBottom":"16px"}),
        dcc.Graph(id="fig3d", style={"height":"85vh"})
    ])

    @app.callback(Output("pilot-dropdown","value"), Input("select-all-btn","n_clicks"), prevent_initial_call=True)
    def select_all_pilots(n_clicks):
        return filenames

    @app.callback(Output("fig3d","figure"), Input("pilot-dropdown","value"))
    def update_figure(selected_pilots):
        fig = go.Figure()
        for name in selected_pilots:
            sub = df_top[df_top["Filename"] == name]
            for tn, gp in sub.groupby("ThermalNumber"):
                fig.add_trace(go.Scatter3d(
                    x=gp["x_plot"],       # centered UTM Easting [m]
                    y=gp["y_plot"],       # centered UTM Northing [m]
                    z=gp["alt"],
                    mode="markers+lines",
                    line=dict(width=2, color="black"),
                    marker=dict(
                        size=3,
                        color=gp[color_field],
                        colorscale="Turbo",
                        cmin=cmin, cmax=cmax,
                        opacity=0.8,
                        showscale=False
                    ),
                    hovertext=[
                        f"<b>{os.path.basename(name)}</b><br>"
                        f"Thermal #{int(t)}<br>"
                        f"TE Rate: {te:.2f} J/kg/s<br>"
                        f"Alt: {alt:.0f} m<br>"
                        f"t={ft:.1f}s"
                        for t, te, alt, ft in zip(
                            gp["ThermalNumber"], gp[color_field], gp["alt"], gp["flighttime"]
                        )
                    ],
                    hoverinfo="text",
                    showlegend=False
                ))

        # shared colorbar (also in meters)
        fig.add_trace(go.Scatter3d(
            x=[df_top["x_plot"].iloc[0]],
            y=[df_top["y_plot"].iloc[0]],
            z=[df_top["alt"].iloc[0]],
            mode="markers",
            marker=dict(
                size=0.1,
                color=[(cmin+cmax)/2],
                colorscale="Turbo",
                cmin=cmin, cmax=cmax,
                colorbar=dict(
                    title=dict(text="Total Energy Rate (J/kg/s)", side="right"),
                    tickformat=".2f",
                    len=0.8,
                    x=1.05
                ),
                showscale=True
            ),
            hoverinfo="none",
            showlegend=False
        ))

        fig.update_layout(
            scene=dict(
                xaxis_title="East (m)",
                yaxis_title="North (m)",
                zaxis_title="Altitude (m)",
                xaxis=dict(range=[float(X_MIN), float(X_MAX)]),
                yaxis=dict(range=[float(Y_MIN), float(Y_MAX)]),
                aspectmode="cube"
            ),
            title=f"3D Flight Paths — Thermal Group {int(top_group)}<br>Colored by Total Energy Rate (J/kg/s)",
            height=850,
            margin=dict(l=0, r=0, b=0, t=60),
            showlegend=False
        )
        return fig


    if RUN_DASH_APP and __name__ == "__main__":
        print("Dash app running — http://127.0.0.1:8092/")
        app.run_server(host="127.0.0.1", port=8092, debug=False)


## 5. Modelling of Thermals

In [ ]:
# Objective 5.1 : Load Utils and mention configuration (Block 1)

for d in (PLOTS_DIR, DATAFRAMES_DIR, SUMMARY_DIR, THERMALLING_EFFICIENCY_DIR):
    d.mkdir(parents=True, exist_ok=True)

def summary_json_path(thermal_group: int) -> str:
    return str(SUMMARY_DIR / f"thermal_group_{int(thermal_group)}_summary.json")

def choose_te_rate(df: pd.DataFrame) -> pd.Series:
    if "TotalEnergy_rate" in df.columns and df["TotalEnergy_rate"].notna().any():
        return pd.to_numeric(df["TotalEnergy_rate"], errors="coerce")
    g = 9.81
    for col in ("TE_vario", "TotalEnergyVario"):
        if col in df.columns and df[col].notna().any():
            return pd.to_numeric(df[col], errors="coerce") * g
    return pd.Series(np.nan, index=df.index)

def utm_xy(lat: np.ndarray, lon: np.ndarray, lat0: float, lon0: float) -> Tuple[np.ndarray, np.ndarray, Dict[str, Any]]:
    lat = np.asarray(lat, float)
    lon = np.asarray(lon, float)

    zone = int(np.floor((lon0 + 180.0) / 6.0) + 1)
    is_north = lat0 >= 0.0
    epsg = 32600 + zone if is_north else 32700 + zone

    crs_utm = pyproj.CRS.from_epsg(epsg)
    crs_ll = pyproj.CRS.from_epsg(4326)
    tfm = pyproj.Transformer.from_crs(crs_ll, crs_utm, always_xy=True)

    x0, y0 = tfm.transform(lon0, lat0)
    x, y = tfm.transform(lon, lat)

    x_rel = x - x0
    y_rel = y - y0

    meta = {"utm_zone": zone, "x_center": float(x0), "y_center": float(y0)}
    return x_rel, y_rel, meta


def detect_circles(df: pd.DataFrame, trim_seconds: float = CIRCLE_TRIM_SECONDS) -> pd.DataFrame:
    g = df.sort_values("flighttime").copy()
    if len(g) < 10 or g["bearing"].isna().all():
        g["CircleNumber"] = np.nan
        return g

    t0 = g["flighttime"].min()
    g = g[g["flighttime"] >= t0 + trim_seconds].copy()
    if len(g) < 10:
        g["CircleNumber"] = np.nan
        return g

    b = np.radians(g["bearing"].to_numpy())
    b_unw = np.unwrap(b) * 180.0 / np.pi
    delta = b_unw - b_unw[0]
    g["CircleNumber"] = np.floor(np.abs(delta) / 360.0).astype(int) + 1
    return g


def deep_merge(a: Dict[str, Any], b: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(a)
    for k, v in b.items():
        if k in out and isinstance(out[k], dict) and isinstance(v, dict):
            out[k] = deep_merge(out[k], v)
        else:
            out[k] = v
    return out


def json_merge_save(path: str, payload: dict, reset: bool = False):
    base = {}
    if not reset and os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                txt = f.read()
            if txt.strip():
                base = json.loads(txt)
        except Exception:
            base = {}

    merged = deep_merge(base, payload)
    tmp = f"{path}.tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(merged, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)
    return merged


def save_html(fig, path: str, label: str = None):
    global PLOT_LOG
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(path, include_plotlyjs="cdn")
    PLOT_LOG.append(path)
    if VERBOSE:
        display(HTML(f'✅ <a href="{path}" target="_blank">{label or Path(path).name}</a>'))


def thermal_model_profiles(r: np.ndarray, R_ref: float) -> Dict[str, np.ndarray]:
    """
    Dimensionless radial profiles for Lenschow / Allen / Childress,
    scaled to a reference radius R_ref and returning shapes (not absolute TE).
    """
    r = np.asarray(r, float)
    if not np.isfinite(R_ref) or R_ref <= 0.0:
        z = np.zeros_like(r, float)
        return {"Lenschow": z, "Allen": z, "Childress": z}

    rho = np.clip(r / R_ref, 0.0, None)

    # Lenschow-like: simple bell-shaped core decaying to 0 at R_ref
    rho_c = np.minimum(rho, 1.0)
    w_l = np.cos(0.5 * np.pi * rho_c)
    w_l[rho >= 1.0] = 0.0
    w_l = np.maximum(0.0, w_l)

    # Allen-like: revolved trapezoid, flat core then linear decay to zero
    rho1 = 0.4
    w_a = np.zeros_like(r, float)
    core = rho <= rho1
    shell = (rho > rho1) & (rho < 1.0)
    w_a[core] = 1.0
    w_a[shell] = 1.0 - (rho[shell] - rho1) / (1.0 - rho1)
    # outside >= 1.0 stays zero

    # Childress-like: central downdraft plus surrounding updraft ring
    rho1c = 0.35
    rho2c = 1.0
    wD = -0.3
    w_c = np.zeros_like(r, float)
    core_c = rho < rho1c
    ring_c = (rho >= rho1c) & (rho <= rho2c)
    if core_c.any():
        w_c[core_c] = wD * np.cos(
            0.5 * np.pi * rho[core_c] / max(rho1c, 1e-6)
        )
    if ring_c.any():
        w_c[ring_c] = np.sin(
            1.212 * np.pi * (rho[ring_c] - rho1c) / max(rho2c - rho1c, 1e-6)
        )

    return {"Lenschow": w_l, "Allen": w_a, "Childress": w_c}

def extract_opt_radius_and_max_efficiency(thermal_group: int) -> Tuple[float, float]:
    """Return (opt_radius_for_thermalling_m, max_efficiency_achieved_pct) for a group."""
    tg = int(thermal_group)
    opt_radius = np.nan
    max_eff_pct = np.nan

    # 1) optimum radius from summary JSON (Block 3 → average_plane → optimal_thermalling.radius_m)
    path_json = summary_json_path(tg)
    if os.path.exists(path_json):
        try:
            with open(path_json, "r", encoding="utf-8") as f:
                js = json.load(f)
            opt_radius = float(
                js
                .get("block3", {})
                .get("average_plane", {})
                .get("optimal_thermalling", {})
                .get("radius_m", np.nan)
            )
        except Exception as e:
            logger.exception("JSON read failed for group %d in extract_opt_radius_and_max_efficiency", tg)

    # 2) max efficiency from per-group thermalling CSV (if enabled)
    if COMPUTE_THERMALLING_EFFICIENCY:
        eff_path = THERMALLING_EFFICIENCY_DIR / f"thermal_group_{tg}_thermalling_efficiency.csv"
        if eff_path.exists():
            try:
                eff_df = pd.read_csv(eff_path)
                col = "thermalling_efficiency_pct"
                if col in eff_df.columns and eff_df[col].notna().any():
                    max_eff_pct = float(eff_df[col].max())
            except Exception as e:
                logger.exception("CSV read failed for group %d in extract_opt_radius_and_max_efficiency", tg)

    return opt_radius, max_eff_pct

if VERBOSE:
    print("Block 1: utils, config and helpers loaded.")


In [ ]:
# Objective 5.2 : Calculate optimum radius for thermalling, centers, drift (Block 2)
# 1. Detect circles for all segments; for each circle compute mean lat/lon/alt/time and TE metrics (mean TE rate and TE gain).
# 2. Rank circles by TE gain, restrict to official time window, and keep top N circles.
# 3. Compute drift path from circle centers in UTM coordinates.
# 4. compute optimum radius for thermalling by calculating the median from all circles.
# Includes df_top filtering and altitude range calculation

# --- df_top cleaning and altitude range for selected thermal group ---

def block2_opt_radius_and_drift() -> None:
    global df_top, centers_ranked, xf, yf, lat0, lon0

    df_top[["lat", "lon", "alt"]] = (
        df_top[["lat", "lon", "alt"]]
        .apply(pd.to_numeric, errors="coerce")
    )

    t0 = df_top.groupby(["Filename", "ThermalNumber"])["time_dt"].transform("min")
    df_top = df_top[(df_top["time_dt"] - t0).dt.total_seconds() >= CIRCLE_TRIM_SECONDS].copy()
    df_top.dropna(subset=["lat", "lon", "alt", "TotalEnergy_rate"], inplace=True)

    if df_top.empty:
        raise ValueError("No data after trimming in Block 2.")

    df_group = df_top.copy()
    if df_group.empty:
        raise ValueError(f"No rows in df_top for thermal_group {top_group} in Block 2.")


    alt_min = float(df_group["alt"].quantile(0.02))
    alt_max = float(df_group["alt"].quantile(0.98))

    # --- robust time parsing for this group ---

    t_strict = pd.to_datetime(df_group["time"], format="%H:%M:%S", errors="coerce")
    mask = t_strict.isna()
    if mask.any():
        t_strict.loc[mask] = pd.to_datetime(df_group.loc[mask, "time"], errors="coerce")

    df_group["time_dt"] = t_strict
    df_group = df_group.dropna(subset=["time_dt"]).sort_values(
        ["Filename", "ThermalNumber", "time_dt"]
    )

    # --- start times within first 10 min ---

    start_times = (
        df_group.groupby(["Filename", "ThermalNumber"], as_index=False)
        .agg(
            start_time=("time_dt", "first"),
            n_points=("time_dt", "size"),
            lat_center=("lat", "mean"),
            lon_center=("lon", "mean"),
        )
        .sort_values("start_time")
    )

    t0_group = start_times["start_time"].min()
    t_cut = t0_group + pd.Timedelta(minutes=10)
    start_times_early = start_times[start_times["start_time"] <= t_cut].copy()

    # --- circle detection on trimmed df_top ---

    df_top = df_top.sort_values(["Filename", "ThermalNumber", "flighttime"])
    df_top = df_top.dropna(subset=["lat", "lon", "alt", "bearing"])

    cr = []
    for (fn, tn), g in df_top.groupby(["Filename", "ThermalNumber"], sort=False):
        g2 = detect_circles(g, trim_seconds=CIRCLE_TRIM_SECONDS)
        g2["Filename"] = fn
        g2["ThermalNumber"] = tn
        cr.append(g2)

    df_circ = pd.concat(cr, ignore_index=True)

    # drop segments where circle detection failed (CircleNumber NaN)
    df_circ = df_circ[df_circ["CircleNumber"].notna()].copy()
    if df_circ.empty:
        raise ValueError(f"No valid circles detected for thermal_group {top_group} in Block 2.")

    # make sure CircleNumber is integer from here on
    df_circ["CircleNumber"] = df_circ["CircleNumber"].astype(int)

    df_circ["File_Thermal_Circle"] = (
        df_circ["Filename"].astype(str)
        + "_"
        + df_circ["ThermalNumber"].astype(int).astype(str)
        + "_"
        + df_circ["CircleNumber"].astype(str)
    )


    # --- circle centers and mean times ---

    rows = []
    for key, g in df_circ.groupby("File_Thermal_Circle", sort=False):
        if len(g) < 5:
            continue

        fn = g["Filename"].iloc[0]
        tn = int(g["ThermalNumber"].iloc[0])
        cn = int(g["CircleNumber"].iloc[0])

        lc = g["lat"].mean()
        lo = g["lon"].mean()
        am = g["alt"].mean()

        if "time_dt" in g.columns and g["time_dt"].notna().any():
            tm = pd.to_datetime(g["time_dt"]).mean()
        elif "time" in g.columns:
            t1 = pd.to_datetime(g["time"], format="%H:%M:%S", errors="coerce")
            t1 = t1.fillna(pd.to_datetime(g["time"], errors="coerce"))
            tm = t1.mean()
        else:
            tm = pd.NaT

        rows.append(
            {
                "File_Thermal_Circle": key,
                "Filename": fn,
                "ThermalNumber": tn,
                "CircleNumber": cn,
                "lat_center": float(lc),
                "lon_center": float(lo),
                "alt_mean": float(am),
                "time_mean_HHMMSS": (tm.strftime("%H:%M:%S") if pd.notna(tm) else ""),
            }
        )

    centers_df = pd.DataFrame(rows)

    # --- TE rate and TE gain per circle, restricted to early window ---

    valid_pairs = set(zip(start_times_early["Filename"], start_times_early["ThermalNumber"]))

    centers_early = centers_df[
        centers_df.apply(lambda r: (r["Filename"], r["ThermalNumber"]) in valid_pairs, axis=1)
    ].copy()

    te_rate_map = df_circ.groupby("File_Thermal_Circle")["TotalEnergy_rate"].mean().to_dict()
    centers_early["te_rate_mean"] = centers_early["File_Thermal_Circle"].map(te_rate_map)

    if "TotalEnergy" not in df_circ.columns:
        raise KeyError("TotalEnergy column missing; needed for TE_gain per circle.")

    te_gain_map = (
        df_circ.groupby("File_Thermal_Circle")["TotalEnergy"]
        .agg(lambda s: s.iloc[-1] - s.iloc[0])
        .to_dict()
    )

    centers_early["te_gain"] = centers_early["File_Thermal_Circle"].map(te_gain_map)

    # --- thermalling efficiency per pilot in this thermal group (guarded by handle) ---

    if COMPUTE_THERMALLING_EFFICIENCY:
        df_circ["te_gain_circle"] = df_circ["File_Thermal_Circle"].map(te_gain_map).astype(float)

        circ_cols = ["File_Thermal_Circle", "Filename", "ThermalNumber", "CircleNumber"]
        if "thermal_group" in df_circ.columns:
            circ_cols.append("thermal_group")

        circ_meta = (
            df_circ
            .drop_duplicates("File_Thermal_Circle")[circ_cols]
            .copy()
        )
        circ_meta["te_gain_circle"] = circ_meta["File_Thermal_Circle"].map(te_gain_map).astype(float)

        if "thermal_group" in circ_meta.columns:
            circ_group = circ_meta[circ_meta["thermal_group"] == top_group].copy()
        else:
            circ_group = circ_meta.copy()
            circ_group["thermal_group"] = int(top_group)

        circ_group = circ_group.dropna(subset=["te_gain_circle"])

        if PILOT_ID_COL not in circ_group.columns:
            raise KeyError(f"PILOT_ID_COL='{PILOT_ID_COL}' not found in circle metadata.")

        if not circ_group.empty:
            eff_df = (
                circ_group
                .groupby(PILOT_ID_COL, as_index=False)
                .agg(
                    n_circles=("te_gain_circle", "size"),
                    te_gain_mean_per_circle=("te_gain_circle", "mean"),
                )
            )

            eff_df["thermal_group"] = int(top_group)
            eff_df["alt_gain_mean_per_circle_m"] = eff_df["te_gain_mean_per_circle"] / G0

            eff_path = THERMALLING_EFFICIENCY_DIR / f"thermal_group_{top_group}_thermalling_efficiency.csv"
            eff_df.to_csv(eff_path, index=False)
            print(f"Thermalling efficiency table saved → {eff_path}")
        else:
            logger.warning("No valid circles for thermalling efficiency in thermal_group = %d.", top_group)


    centers_ranked = (
        centers_early.assign(te_gain=centers_early["te_gain"].astype(float))
        .nlargest(TOP_N_CIRCLES, "te_gain")
    )

    centers_ranked["time_dt"] = pd.to_datetime(
        centers_ranked["time_mean_HHMMSS"], format="%H:%M:%S", errors="coerce"
    )

    centers_ranked = centers_ranked.sort_values("time_dt").dropna(
        subset=["lat_center", "lon_center", "time_dt"]
    )

    if centers_ranked.empty:
        raise ValueError("No centers available for drift in Block 2.")

    # --- UTM drift path ---

    lat0 = centers_ranked["lat_center"].mean()
    lon0 = centers_ranked["lon_center"].mean()

    x, y, _ = utm_xy(
        centers_ranked["lat_center"].to_numpy(),
        centers_ranked["lon_center"].to_numpy(),
        lat0,
        lon0,
    )

    # time axis in seconds from first center
    t = (centers_ranked["time_dt"] - centers_ranked["time_dt"].iloc[0]).dt.total_seconds().to_numpy()
    x = x.astype(float)
    y = y.astype(float)

    # drop duplicate times (just in case)
    t , idx = np.unique(t , return_index=True)
    x = x[idx]
    y = y[idx]

    n = len(t)
    if n < 2:
        raise ValueError(f"Not enough centers for drift spline in group {top_group} (need ≥2, got {n}).")

    # first try original cubic spline; if that fails, fall back to lower order
    sf = n * 5e5
    try:
        # your original intent
        xf = UnivariateSpline(t , x , k=3 , s=sf)
        yf = UnivariateSpline(t , y , k=3 , s=sf)
    except Exception as e:
        # fallback based on number of points: n=3→k=2, n=2→k=1
        k = 3 if n > 3 else n - 1
        # if that still fails, let it raise so the group is skipped above
        xf = UnivariateSpline(t , x , k=k , s=0)
        yf = UnivariateSpline(t , y , k=k , s=0)

    t_f = np.linspace(t.min(), t.max(), 300)
    x_s = xf(t_f)
    y_s = yf(t_f)

    d = np.sqrt(np.diff(x_s) ** 2 + np.diff(y_s) ** 2)
    cum_d = np.concatenate(([0.0], np.cumsum(d)))

    # --- TE rate vs distance along drift ---

    dist_pts = np.interp(x, x_s, cum_d)
    te_pts = centers_ranked["te_rate_mean"].to_numpy()

    te_df = (
        pd.DataFrame(
            {"dist_m": dist_pts.astype(float), "te_rate_mean": te_pts.astype(float)}
        )
        .dropna()
        .sort_values("dist_m")
    )

    x_te = te_df["dist_m"].to_numpy()
    y_te = te_df["te_rate_mean"].to_numpy()
    n_te = x_te.size

    if n_te >= 4:
        # sort by x
        idx = np.argsort(x_te)
        x_te = x_te[idx]
        y_te = y_te[idx]

        # remove (almost) duplicate x to keep x strictly increasing
        keep = np.r_[True, np.diff(x_te) > 1e-6]
        x_spl = x_te[keep]
        y_spl = y_te[keep]

        if x_spl.size >= 3:
            # ORIGINAL APPROACH: spline fit (only when valid)
            var_y = float(np.var(y_spl)) if x_spl.size > 1 else 1.0
            s_spl = 5.0 * var_y * x_spl.size

            try:
                sp = UnivariateSpline(x_spl, y_spl, k=2, s=s_spl)
                dm = np.linspace(x_spl.min(), x_spl.max(), 300)
                dm_y = sp(dm)
            except Exception:
                # if SciPy still complains for some corner case, fall back
                dm = np.linspace(x_spl.min(), x_spl.max(), 200)
                dm_y = np.interp(dm, x_spl, y_spl)

        elif x_spl.size >= 2:
            # too few points for spline, but interpolation is fine
            dm = np.linspace(x_spl.min(), x_spl.max(), 200)
            dm_y = np.interp(dm, x_spl, y_spl)
        else:
            # effectively no usable points
            dm = np.array([])
            dm_y = np.array([])

    elif n_te >= 2:
        # not enough points for any smoothing; simple interpolation
        dm = np.linspace(x_te.min(), x_te.max(), 200)
        dm_y = np.interp(dm, x_te, y_te)
    else:
        dm = np.array([])
        dm_y = np.array([])


    # --- plot drift and TE rate vs distance ---

    fig = make_subplots(
        rows=2,
        cols=1,
        vertical_spacing=0.12,
        subplot_titles=("Core Drift (UTM meters)", "TE rate vs Drift Distance"),
    )

    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="markers+text",
            text=centers_ranked["time_mean_HHMMSS"],
            textposition="top center",
            name="Centers",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=te_df["dist_m"],
            y=te_df["te_rate_mean"],
            mode="markers",
            name="TE rate",
        ),
        row=2,
        col=1,
    )

    if dm.size:
        fig.add_trace(
            go.Scatter(
                x=dm,
                y=dm_y,
                mode="lines",
                line=dict(width=3),
                name="Spline/Interp",
            ),
            row=2,
            col=1,
        )

    fig.add_trace(
        go.Scatter(x=x_s, y=y_s, mode="lines", name="Smoothed Drift"),
        row=1,
        col=1,
    )

    fig.update_xaxes(title_text="East [m]", row=1, col=1)
    fig.update_yaxes(title_text="North [m]", row=1, col=1, scaleanchor="x", scaleratio=1)
    fig.update_xaxes(title_text="Distance [m]", row=2, col=1)
    fig.update_yaxes(title_text="TE rate [J/kg/s]", row=2, col=1)

    fig.update_layout(
        height=900,
        showlegend=True,
        title=f"Group {top_group} — Drift and TE rate",
    )

    drift_html = None
    if PLOT_DRIFT_TE_RATE:
        drift_html = f"Results/plots/drift_te_rate/thermal_group_{top_group}_drift_te_rate.html"
        save_html(fig, drift_html, label="Drift and TE rate")

    # --- per-circle radius stats and geometry payload ---


    def circle_radius_stats(g: pd.DataFrame) -> tuple[float, float, int]:
        latc = float(g["lat"].mean())
        lonc = float(g["lon"].mean())
        xg, yg, _ = utm_xy(g["lat"].to_numpy(), g["lon"].to_numpy(), latc, lonc)
        rad = np.hypot(xg, yg)
        return float(np.nanmean(rad)), float(np.nanstd(rad)), int(rad.size)


    rad_rows = []
    for key, g in df_circ.groupby("File_Thermal_Circle", sort=False):
        if len(g) < 5:
            continue
        r_mean, r_std, n = circle_radius_stats(g)
        rad_rows.append((key, r_mean, r_std, n))

    rad_df = pd.DataFrame(
        rad_rows,
        columns=["File_Thermal_Circle", "radius_mean_m", "radius_std_m", "n_points_circle"],
    )

    centers_df = centers_df.merge(rad_df, on="File_Thermal_Circle", how="left")

    valid_circles = centers_ranked["File_Thermal_Circle"].unique()
    centers_sel = centers_df[centers_df["File_Thermal_Circle"].isin(valid_circles)].copy()

    z_start_m = alt_min
    z_end_m = alt_max
    R_mean_m = float(np.nanmedian(centers_sel["radius_mean_m"]))

    start_times_early_json = start_times_early.assign(
        start_time=start_times_early["start_time"].astype(str)
    )[["Filename", "ThermalNumber", "start_time", "n_points"]].to_dict(orient="records")

    centers_summary_json = centers_df.to_dict(orient="records")

    t0_dt = centers_ranked["time_dt"].iloc[0]
    t_hms = [
        (t0_dt + pd.to_timedelta(s, unit="s")).strftime("%H:%M:%S")
        for s in t_f
    ]

    drift_json = {
        "drift_ref_latlon": {"lat0": float(lat0), "lon0": float(lon0)},
        "drift_path_xy_m": {
            "t_hms": t_hms,
            "x_m": x_s.tolist(),
            "y_m": y_s.tolist(),
            "cum_dist_m": cum_d.tolist(),
        },
        "te_rate_vs_distance": te_df.to_dict(orient="records"),
        "plot_html": drift_html,
    }

    geom_payload = {
        "geometry": {
            "z_start_m": float(z_start_m),
            "z_end_m": float(z_end_m),
            "radius_mean_m": R_mean_m,
        }
    }

    payload_block2 = {
        "block2": {
            "thermal_group": int(top_group),
            "start_times_early": start_times_early_json,
            "circle_centers_summary": centers_summary_json,
            "drift": drift_json,
            **geom_payload,
        }
    }

    summary = json_merge_save(
        summary_json_path(top_group),
        payload_block2,
        reset=True,        # start a fresh summary for this group
    )
    if VERBOSE:
        print(f"✅ Block 2 (drift / optimum radius) stored → {summary_json_path(top_group)}")


In [ ]:
# Objective 5.3 : Strength distribuition calculation (Block 3)
# 1. Trim the first 10 seconds of each thermal
# 2. Calculate the global center and height range
# 3. Analyze thermal strength distribution in horizontal xy slices
# 4. Compute TE vs altitude profile and a smoothed (rolling) TE vs altitude.
# 5. Compute heading from motion, then a smoothed TE vs heading profile.

def block3_strength_distribution() -> None:
    global df_top

    te_min, te_max = TE_MIN, TE_MAX

    num_slices = NUM_SLICES
    slice_half_thickness0 = SLICE_HALF_THICKNESS0
    slice_half_thickness_max = SLICE_HALF_THICKNESS_MAX
    min_points_per_slice = MIN_POINTS_PER_SLICE
    radius_rings_m = RADIUS_RINGS_M
    rank_colors = RANK_COLORS

    df_top[["lat", "lon", "alt"]] = (
        df_top[["lat", "lon", "alt"]]
        .apply(pd.to_numeric, errors="coerce")
    )

    df_top.dropna(subset=["lat", "lon", "alt", "TotalEnergy_rate", "time_dt"], inplace=True)
    if df_top.empty:
        raise ValueError("No data after cleaning in Block 3.")

    # -------------------------------------------------------------
    # drift-corrected coordinates (use drift model from previous block)
    # -------------------------------------------------------------

    # use same UTM reference as in drift block
    center_lat = float(lat0)
    center_lon = float(lon0)

    # raw UTM coordinates relative to (lat0, lon0)
    x_raw, y_raw, meta = utm_xy(
        df_top["lat"].to_numpy(),
        df_top["lon"].to_numpy(),
        center_lat,
        center_lon,
    )

    # time offset [s] of each sample from drift reference t0_dt
    t0_dt_ref = centers_ranked["time_dt"].iloc[0]
    t_all = (df_top["time_dt"] - t0_dt_ref).dt.total_seconds().to_numpy()

    # time range used for drift spline; clip samples to this range
    t_centers = (centers_ranked["time_dt"] - t0_dt_ref).dt.total_seconds().to_numpy()
    t_min_s = float(t_centers.min())
    t_max_s = float(t_centers.max())
    t_all_clip = np.clip(t_all, t_min_s, t_max_s)

    # smoothed core position from drift splines
    x_core = xf(t_all_clip)
    y_core = yf(t_all_clip)

    # drift-removed local coordinates (core at x=y=0)
    df_top["x_m"] = x_raw - x_core
    df_top["y_m"] = y_raw - y_core
    df_top["radius_m"] = np.hypot(df_top["x_m"], df_top["y_m"])

    axis_limit = 1.05 * max(abs(df_top["x_m"]).max(), abs(df_top["y_m"]).max())
    zmin = float(df_top["alt"].min())
    zmax = float(df_top["alt"].max())

    alts = np.quantile(df_top["alt"], np.linspace(0.15, 0.85, num_slices))

    fig_xy = make_subplots(
        rows=1,
        cols=num_slices,
        horizontal_spacing=0.04,
        subplot_titles=[f"Alt ~ {int(a)} m" for a in alts],
    )

    xy_slices_records = []
    theta = np.linspace(0.0, 2.0 * np.pi, 240)
    slice_ring_means = []

    for i, a in enumerate(alts, start=1):
        dz = slice_half_thickness0
        while True:
            sl = df_top[(df_top["alt"] >= a - dz) & (df_top["alt"] <= a + dz)]
            for name , g in sl.groupby("Filename"):
                fig_xy.add_trace(
                    go.Scatter(
                        x=g["x_m"],
                        y=g["y_m"],
                        mode="lines",
                        line=dict(width=1),
                        name=name,
                        hoverinfo="skip",
                        showlegend=False,
                    ),
                    row=1,
                    col=i,
                )
            if (len(sl) >= min_points_per_slice) or (dz >= slice_half_thickness_max):
                break
            dz *= 1.5

        fig_xy.add_trace(
            go.Scatter(
                x=sl["x_m"],
                y=sl["y_m"],
                mode="markers",
                marker=dict(
                    size=5,
                    color=sl["TotalEnergy_rate"],
                    cmin=te_min,
                    cmax=te_max,
                    colorscale=CFD_COLORSCALE,
                    colorbar=dict(title="TE rate (J/kg/s)") if i == num_slices else None,
                    opacity=0.9,
                ),
                text=sl["Filename"],
                hovertemplate=(
                    "x:%{x:.1f} m<br>"
                    "y:%{y:.1f} m<br>"
                    "alt:%{customdata[0]:.1f} m<br>"
                    "time:%{customdata[1]}<br>"
                    "TE rate:%{marker.color:.1f} J/kg/s<br>"
                    "pilot:%{text}<extra></extra>"
                ),
                customdata=np.c_[sl["alt"].to_numpy(), sl["time"].astype(str).to_numpy()],
                showlegend=False,
            ),
            row=1,
            col=i,
        )

        ring_means = []
        for rr in radius_rings_m:
            m = (sl["radius_m"] >= rr - 10.0) & (sl["radius_m"] <= rr + 10.0)
            ring_means.append(sl.loc[m, "TotalEnergy_rate"].mean() if m.any() else np.nan)
            xy_slices_records.append(
                {
                    "slice_alt_center_m": float(a),
                    "slice_half_thickness_m": float(dz),
                    "ring_r_m": float(rr),
                    "ring_te_rate_mean": float(ring_means[-1]) if np.isfinite(ring_means[-1]) else np.nan,
                    "n_points_slice": int(len(sl)),
                }
            )

        slice_ring_means.append(ring_means.copy())

        fig_xy.add_trace(
            go.Scatter(
                x=[0.0],
                y=[0.0],
                mode="markers+text",
                marker=dict(size=10, symbol="x", line=dict(width=2)),
                text=["center"],
                textposition="top center",
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=i,
        )

        order = np.argsort(np.array(ring_means))[::-1]
        for k, idx in enumerate(order):
            rr = radius_rings_m[idx]
            color = rank_colors[min(k, len(rank_colors) - 1)]
            fig_xy.add_trace(
                go.Scatter(
                    x=rr * np.cos(theta),
                    y=rr * np.sin(theta),
                    mode="lines",
                    line=dict(width=3, color=color),
                    hoverinfo="skip",
                    showlegend=False,
                ),
                row=1,
                col=i,
            )

        fig_xy.update_xaxes(
            title_text="East (m)",
            range=[-axis_limit, axis_limit],
            row=1,
            col=i,
            scaleanchor=f"y{i}",
            scaleratio=1,
        )
        fig_xy.update_yaxes(
            title_text="North (m)",
            range=[-axis_limit, axis_limit],
            row=1,
            col=i,
        )

    xy_html = None
    fig_xy.update_layout(
        title=(
            f"XY slices — Group {top_group} "
            f"(center {center_lat:.5f}, {center_lon:.5f}, zone {meta['utm_zone']})"
        ),
        height=700,
        margin=dict(l=10, r=10, t=70, b=10),
    )

    if PLOT_XY_SLICES_TERATE:
        xy_html = f"Results/plots/xy_slices_terate/xy_slices_terate_group_{top_group}.html"
        save_html(fig_xy, xy_html)

    # --- mean radial profile on mean plane (new) ---

    # use full thermal: project whole TE distribution onto a mean plane
    mean_alt_plane = float(df_top["alt"].mean())  # just for reference/metadata
    df_plane = df_top.copy()

    # radius of spread from data (95 % quantile of radius)
    spread_radius = float(df_plane["radius_m"].quantile(0.95))
    if not np.isfinite(spread_radius) or spread_radius <= 0.0:
        spread_radius = float(df_plane["radius_m"].max())

    # limit extent of plot: cover spread and where we have data
    r_max_data = float(df_plane["radius_m"].quantile(0.98))
    r_max_plot = float(min(axis_limit, max(r_max_data, 1.5 * spread_radius)))

    # --- radial summary (for JSON only, not for the surface) ---
    n_r_bins = 80
    r_edges = np.linspace(0.0, r_max_plot, n_r_bins + 1)
    r_centers = 0.5 * (r_edges[:-1] + r_edges[1:])

    labels_r = pd.cut(
        df_plane["radius_m"],
        r_edges,
        labels=False,
        include_lowest=True,
    )
    te_by_bin = df_plane.groupby(labels_r)["TotalEnergy_rate"].mean()

    te_r = te_by_bin.reindex(range(n_r_bins)).to_numpy()
    te_r = pd.Series(te_r).interpolate(limit_direction="both").to_numpy()

    avg_plane_profile = [
        {"radius_m": float(r), "te_rate_mean": float(v)}
        for r, v in zip(r_centers, te_r)
    ]

    # --- 2-sigma radius from radial TE distribution (updraft only) ---

    te_pos = np.maximum(0.0, te_r)
    if np.isfinite(te_pos).any() and np.nansum(te_pos) > 0.0:
        num = float(np.nansum(te_pos * (r_centers ** 2)))
        den = float(np.nansum(te_pos))
        var_r = num / max(den, 1e-12)
        sigma_r = float(np.sqrt(max(var_r, 0.0)))
        radius_2sigma = 2.0 * sigma_r
    else:
        sigma_r = float("nan")
        radius_2sigma = float("nan")

    # reference radius for analytic models: prefer 2σ, fall back to spread_radius
    R_ref_models = radius_2sigma if np.isfinite(radius_2sigma) and radius_2sigma > 0.0 else spread_radius
    R_ref_models = float(min(R_ref_models, r_max_plot))

    # --- compare mean-plane radial profile to analytic models ---

    model_fits = []
    model_profiles = {}
    comparison_html = None

    if np.isfinite(R_ref_models) and R_ref_models > 0.0:
        mask_eval = r_centers <= R_ref_models
        r_eval = r_centers[mask_eval]
        data_eval = te_r[mask_eval]

        if r_eval.size >= 5 and np.isfinite(data_eval).any():
            data_eval_pos = np.maximum(0.0, data_eval)
            if np.nanmax(data_eval_pos) > 0.0:
                data_shape = data_eval_pos / np.nanmax(data_eval_pos)

                model_shapes = thermal_model_profiles(r_eval, R_ref_models)

                for name , m_raw in model_shapes.items():
                    m_raw = np.asarray(m_raw, float)
                    m_pos = np.maximum(0.0, m_raw)
                    if (not np.isfinite(m_pos).any()) or np.nanmax(m_pos) <= 0.0:
                        rmse = float("nan")
                        match_pct = float("nan")
                    else:
                        m_shape = m_pos / np.nanmax(m_pos)
                        diff = data_shape - m_shape
                        rmse = float(np.sqrt(np.nanmean(diff ** 2)))
                        match_pct = float(max(0.0, 100.0 * (1.0 - rmse)))

                    model_fits.append(
                        {"model": name, "rmse_norm": rmse, "match_percent": match_pct}
                    )
                    model_profiles[name] = [
                        {"radius_m": float(rr), "shape": float(ss)}
                        for rr, ss in zip(r_eval, m_raw)
                    ]

                if VERBOSE:
                    print("Mean-plane TE radial profile vs analytic models:")
                    for mf in model_fits:
                        if np.isfinite(mf["match_percent"]):
                            print(f" {mf['model']}: {mf['match_percent']:.1f}% match (rmse={mf['rmse_norm']:.3f})")


                # --- comparison plot (like literature figure) ---
                if PLOT_MODEL_COMPARISON:
                    # symmetric distance axis for nicer look
                    x_sym = np.concatenate((-r_eval[::-1], r_eval))

                    # normalized measured profile
                    y_data = data_shape
                    y_data_sym = np.concatenate((y_data[::-1], y_data))

                    fig_comp = go.Figure()
                    fig_comp.add_trace(
                        go.Scatter(
                            x=x_sym,
                            y=y_data_sym,
                            mode="lines",
                            line=dict(width=4),
                            name="Measured (TE-based)",
                        )
                    )

                    for name , prof in model_profiles.items():
                        s_raw = np.array([p["shape"] for p in prof], float)
                        if not np.isfinite(s_raw).any() or np.nanmax(np.abs(s_raw)) <= 0.0:
                            continue
                        # scale so positive peak is 1.0
                        s_pos = np.maximum(0.0, s_raw)
                        if np.nanmax(s_pos) > 0.0:
                            s_plot = s_raw / np.nanmax(s_pos)
                        else:
                            s_plot = s_raw / np.nanmax(np.abs(s_raw))
                        y_sym = np.concatenate((s_plot[::-1], s_plot))
                        fig_comp.add_trace(
                            go.Scatter(
                                x=x_sym,
                                y=y_sym,
                                mode="lines",
                                line=dict(width=2),
                                name=name,
                            )
                        )

                    fig_comp.update_layout(
                        title=(f"Radial strength profile vs thermal models — Group {top_group}"),
                        xaxis=dict(title="Distance from center [m]"),
                        yaxis=dict(title="Normalized strength (TE-based)"),
                        height=500,
                        legend=dict(orientation="h", yanchor="bottom", y=-0.2),
                        margin=dict(l=40, r=40, t=60, b=80),
                    )

                    comparison_html = (
                        f"Results/plots/comparisions_to_other_thermal_models/"
                        f"thermal_group_{top_group}_model_comparison.html"
                    )
                    save_html(fig_comp, comparison_html, label="Thermal model comparison")


    # --- 2D gridded TE distribution for the surface ---

    # grid extents (square around center)
    x_min = -r_max_plot
    x_max =  r_max_plot
    y_min = -r_max_plot
    y_max =  r_max_plot

    n_x = 80
    n_y = 80

    x_edges = np.linspace(x_min, x_max, n_x + 1)
    y_edges = np.linspace(y_min, y_max, n_y + 1)
    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])

    # mean TE in each (x,y) cell via weighted 2D histogram
    H_sum, _, _ = np.histogram2d(
        df_plane["x_m"], df_plane["y_m"],
        bins=[x_edges, y_edges],
        weights=df_plane["TotalEnergy_rate"],
    )
    H_cnt, _, _ = np.histogram2d(
        df_plane["x_m"], df_plane["y_m"],
        bins=[x_edges, y_edges],
    )

    te_grid = np.divide(
        H_sum,
        H_cnt,
        out=np.full_like(H_sum, np.nan, dtype=float),
        where=H_cnt > 0,
    )

    # neighbour-mean inpainting + Gaussian smoothing
    if np.isfinite(te_grid).any():
        mask = np.isfinite(te_grid).astype(float)
        te_filled = np.where(np.isfinite(te_grid), te_grid, 0.0)

        te_smooth = gaussian_filter(te_filled, sigma=1.5)
        mask_smooth = gaussian_filter(mask, sigma=1.5)

        with np.errstate(invalid="ignore", divide="ignore"):
            te_grid_smooth = te_smooth / np.maximum(mask_smooth, 1e-6)
    else:
        te_grid_smooth = te_grid

    Xg, Yg = np.meshgrid(x_centers, y_centers, indexing="ij")
    R_grid = np.hypot(Xg, Yg)

    # start from smoothed interior field
    Zg = te_grid_smooth.copy()

    # --- smooth Gaussian decay to zero (no hard seam) ---

    R0 = spread_radius
    R1 = r_max_plot
    eps_tail = 1e-3                           # value at R1 ~ 0.001 of center

    delta_r = max(R1 - R0, 1e-3)
    sigma_outer = delta_r / np.sqrt(2.0 * np.log(1.0 / eps_tail))

    radial_weight = np.ones_like(R_grid)
    mask_outer = R_grid > R0
    radial_weight[mask_outer] = np.exp(
        -((R_grid[mask_outer] - R0) ** 2) / (2.0 * sigma_outer ** 2)
    )
    radial_weight[R_grid >= R1] = 0.0

    Zg *= radial_weight
    Zg = np.clip(Zg, TE_MIN_2, TE_MAX_2)
    Zg *= radial_weight
    Zg = np.clip(Zg, TE_MIN_2, TE_MAX_2)

    # --- optimal thermalling circle from mean-plane TE field ---

    opt_radius_m = np.nan
    opt_te_gain_per_circle_ref = np.nan
    opt_te_rate_mean = np.nan
    opt_circle_profile = None

    if np.isfinite(Zg).any():
        try:
            interp = RegularGridInterpolator(
                (x_centers, y_centers),
                Zg,
                bounds_error=False,
                fill_value=np.nan,
            )

            r_start = max(MIN_TURN_RADIUS_M, 0.1 * r_max_plot)
            r_end   = r_max_plot
            if r_end <= 0.0:
                r_start, r_end = 0.0, 0.0

            r_candidates = np.linspace(r_start, r_end, 50)

            theta_path = np.linspace(0.0, 2.0 * np.pi, 360)
            te_gain_model = []
            te_rate_means = []
            circle_profiles = []

            for r in r_candidates:
                xs = r * np.cos(theta_path)
                ys = r * np.sin(theta_path)
                pts = np.column_stack((xs, ys))
                vals = interp(pts)

                if not np.isfinite(vals).any():
                    te_gain_model.append(np.nan)
                    te_rate_means.append(np.nan)
                    circle_profiles.append(None)
                    continue

                mean_rate = float(np.nanmean(vals))  # J/kg/s along the circle
                te_rate_means.append(mean_rate)

                circle_time = 2.0 * np.pi * r / max(THERMALLING_REFERENCE_TAS_MS, 1e-3)
                te_gain_model.append(mean_rate * circle_time)  # J/kg (model)

                circle_profiles.append((xs, ys, vals))

            te_gain_arr = np.asarray(te_gain_model, float)
            if np.isfinite(te_gain_arr).any():
                idx_best = int(np.nanargmax(te_gain_arr))
                opt_radius_m = float(r_candidates[idx_best])
                opt_te_gain_per_circle_ref = float(te_gain_arr[idx_best])
                opt_te_rate_mean = float(te_rate_means[idx_best])
                opt_circle_profile = circle_profiles[idx_best]
        except Exception as e:
            if VERBOSE:
                print(f"Optimal thermalling circle computation failed: {e}")


    fig_plane = go.Figure(data=[
        go.Surface(
            x=Xg,
            y=Yg,
            z=Zg,
            surfacecolor=Zg,
            colorscale=CFD_COLORSCALE,
            cmin=TE_MIN_2,
            cmax=TE_MAX_2,
            colorbar=dict(title="TE rate (J/kg/s)"),
            hovertemplate=("East:%{x:.1f} m<br>"
                           "North:%{y:.1f} m<br>"
                           "TE rate:%{z:.2f} J/kg/s<extra></extra>"),
        )
    ])


    # mark radius of spread on a base plane
    th = np.linspace(0.0, 2.0 * np.pi, 200)
    x_th = spread_radius * np.cos(th)
    y_th = spread_radius * np.sin(th)
    z_base = np.nanmin(Zg) if np.isfinite(Zg).any() else 0.0
    z_th = np.full_like(x_th, z_base)

    fig_plane.add_trace(
        go.Scatter3d(
            x=x_th,
            y=y_th,
            z=z_th,
            mode="lines",
            line=dict(color="black", width=5),
            name=f"Radius of spread ~{spread_radius:.0f} m",
            hoverinfo="skip",
        )
    )

    if opt_circle_profile is not None:
        xs_opt, ys_opt, vals_opt = opt_circle_profile
        fig_plane.add_trace(
            go.Scatter3d(
                x=xs_opt,
                y=ys_opt,
                z=vals_opt,
                mode="lines",
                line=dict(width=5),
                name=f"Optimum thermalling circle ~{opt_radius_m:.0f} m",
                hoverinfo="skip",
            )
        )

    fig_plane.update_layout(
        title=(f"TE rate distribution on mean plane — Group {top_group} "
               f"(alt slab centered ~{mean_alt_plane:.0f} m)"),
        scene=dict(
            xaxis=dict(title="East (m)", range=[x_min, x_max]),
            yaxis=dict(title="North (m)", range=[y_min, y_max]),
            zaxis=dict(title="TE rate (J/kg/s)", range=[TE_MIN_2, TE_MAX_2]),
            aspectmode="cube",
        ),
        margin=dict(l=10, r=10, t=70, b=10),
    )

    avg_plane_html = None
    if PLOT_MEAN_PLANE_XY_TERATE:
        avg_plane_html = (
            f"Results/plots/mean_plane_xy_terate/"
            f"mean_plane_xy_terate_group_{top_group}.html"
        )
        save_html(fig_plane, avg_plane_html)

    # --- end mean-plane block ---





    # --- 3D flight paths (now drift-corrected in x_m, y_m) ---

    fig3d = go.Figure()

    for name, g in df_top.groupby("Filename"):
        fig3d.add_trace(
            go.Scatter3d(
                x=g["x_m"],
                y=g["y_m"],
                z=g["alt"],
                mode="lines",
                name=name,
                line=dict(width=3),
                hovertext=[
                    f"{name}<br>alt {zz:.1f} m<br>TE rate {cc:.1f} J/kg/s"
                    for zz, cc in zip(g["alt"], g["TotalEnergy_rate"])
                ],
                hoverinfo="text",
            )
        )

    fig3d.add_trace(
        go.Scatter3d(
            x=[0.0, 0.0],
            y=[0.0, 0.0],
            z=[zmin, zmax],
            mode="lines",
            line=dict(width=6, dash="dash"),
            name="Thermal center",
        )
    )

    fig3d.update_layout(
        scene=dict(
            xaxis=dict(title="East (m)"),
            yaxis=dict(title="North (m)"),
            zaxis=dict(title="Altitude (m)", range=[zmin, zmax]),
            aspectmode="cube",
        ),
        title=f"3D Flight Paths — Group {top_group}",
        height=800,
    )

    threeD_html = None
    if PLOT_XY_SLICES_TERATE:     # or define a separate flag if you want
        threeD_html = f"Results/plots/paths_3d/paths_3d_with_center_group_{top_group}.html"
        save_html(fig3d, threeD_html)

    # --- vertical profile of TE rate vs altitude ---

    alt_bins = np.linspace(df_top["alt"].quantile(0.05), df_top["alt"].quantile(0.95), 20)
    alt_mid = 0.5 * (alt_bins[:-1] + alt_bins[1:])

    grp_alt = pd.cut(df_top["alt"], alt_bins)
    vert_profile = (
        df_top.groupby(grp_alt)["TotalEnergy_rate"].mean().reset_index(drop=True)
    )

    vertical_profile = [
        {"alt_m": float(a), "te_rate_mean": float(w)}
        for a, w in zip(alt_mid, vert_profile)
    ]

    # --- rolling strength vs altitude and heading ---

    alt_min_local = float(df_top["alt"].quantile(0.02))
    alt_max_local = float(df_top["alt"].quantile(0.98))
    alt_step = 20.0

    alt_edges = np.arange(
        np.floor(alt_min_local),
        np.ceil(alt_max_local) + alt_step,
        alt_step,
    )
    alt_centers = 0.5 * (alt_edges[:-1] + alt_edges[1:])

    labels_alt = pd.cut(df_top["alt"], alt_edges, labels=False, include_lowest=True)
    alt_mean = (
        pd.Series(df_top["TotalEnergy_rate"])
        .groupby(labels_alt)
        .mean()
        .reindex(range(len(alt_centers)))
    )
    alt_mean = alt_mean.interpolate(limit_direction="both")

    alt_sigma = 2.0
    alt_smooth = gaussian_filter1d(
        alt_mean.to_numpy(),
        sigma=alt_sigma,
        mode="nearest",
    )

    alt_roll_records = [
        {"alt_m": float(a), "te_rate_roll": float(w)}
        for a, w in zip(alt_centers, alt_smooth)
    ]


    def compute_heading(df_top_one: pd.DataFrame) -> pd.DataFrame:
        g = df_top_one.sort_values("time_dt").copy()
        g["dx"] = g["x_m"].diff()
        g["dy"] = g["y_m"].diff()
        g["heading_deg"] = (np.degrees(np.arctan2(g["dx"], g["dy"])) % 360.0)
        return g


    df_top_head = df_top.groupby("Filename", group_keys=False).apply(compute_heading)
    df_top_head = df_top_head.dropna(subset=["heading_deg", "TotalEnergy_rate"])

    hdg_step = 10.0
    hdg_edges = np.arange(0.0, 360.0 + hdg_step, hdg_step)
    hdg_centers = 0.5 * (hdg_edges[:-1] + hdg_edges[1:])

    hdg_bin = (
        np.floor(df_top_head["heading_deg"] / hdg_step)
        .astype(int)
        .clip(0, len(hdg_centers) - 1)
    )

    hdg_mean = (
        pd.Series(df_top_head["TotalEnergy_rate"])
        .groupby(hdg_bin)
        .mean()
        .reindex(range(len(hdg_centers)))
    )
    hdg_mean = hdg_mean.interpolate(limit_direction="both")

    hdg_sigma_bins = 2.0
    hdg_smooth = gaussian_filter1d(
        hdg_mean.to_numpy(),
        sigma=hdg_sigma_bins,
        mode="wrap",
    )

    heading_roll_records = [
        {"heading_deg": float(h), "te_rate_roll": float(w)}
        for h, w in zip(hdg_centers, hdg_smooth)
    ]

    fig_roll = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Rolling TE rate vs altitude", "Rolling TE rate vs heading"),
    )

    fig_roll.add_trace(
        go.Scatter(
            x=alt_centers,
            y=alt_smooth,
            mode="lines",
            line=dict(width=3),
        ),
        row=1,
        col=1,
    )

    fig_roll.update_xaxes(title_text="Altitude [m]", row=1, col=1)
    fig_roll.update_yaxes(title_text="TE rate [J/kg/s]", row=1, col=1)

    fig_roll.add_trace(
        go.Scatter(
            x=hdg_centers,
            y=hdg_smooth,
            mode="lines",
            line=dict(width=3),
        ),
        row=1,
        col=2,
    )

    fig_roll.update_xaxes(title_text="Heading [deg]", row=1, col=2)
    fig_roll.update_yaxes(title_text="TE rate [J/kg/s]", row=1, col=2)

    fig_roll.update_layout(
        height=450,
        margin=dict(l=40, r=40, t=60, b=40),
        showlegend=False,
    )

    roll_html = None
    if PLOT_ROLLING_TE_PROFILES:
        roll_html = (
            f"Results/plots/rolling_TotalEnergy_rate_profiles/"
            f"rolling_TotalEnergy_rate_profiles_group_{top_group}.html"
        )
        save_html(fig_roll, roll_html, label="Rolling TE rate profiles")

    # --- optimal thermalling metrics for JSON and CSV ---

    optimal_thermalling_json = {
        "radius_m": float(opt_radius_m) if np.isfinite(opt_radius_m) else None,
        "te_rate_mean_J_kg_s": float(opt_te_rate_mean) if np.isfinite(opt_te_rate_mean) else None,
        "te_gain_per_circle_ref_J_kg": float(opt_te_gain_per_circle_ref) if np.isfinite(opt_te_gain_per_circle_ref) else None,
        "min_turn_radius_m": float(MIN_TURN_RADIUS_M),
        "reference_tas_mps": float(THERMALLING_REFERENCE_TAS_MS),
    }

    if COMPUTE_THERMALLING_EFFICIENCY and np.isfinite(opt_te_gain_per_circle_ref) and opt_te_gain_per_circle_ref > 0.0:
        eff_path = THERMALLING_EFFICIENCY_DIR / f"thermal_group_{top_group}_thermalling_efficiency.csv"
        if eff_path.exists():
            eff_df = pd.read_csv(eff_path)
            if "te_gain_mean_per_circle" in eff_df.columns:
                eff_df["te_gain_opt_per_circle_ref"] = opt_te_gain_per_circle_ref
                eff_df["alt_gain_opt_per_circle_ref_m"] = opt_te_gain_per_circle_ref / G0
                eff_df["thermalling_efficiency_pct"] = (
                    100.0 * eff_df["te_gain_mean_per_circle"] / opt_te_gain_per_circle_ref
                )
                eff_df.to_csv(eff_path, index=False)
                if VERBOSE:
                    print(f"Updated thermalling efficiency % in → {eff_path}")

                else:
                    logger.warning("'te_gain_mean_per_circle' missing in %s, skipping %% calculation.", eff_path)

        else:
            logger.warning("No thermalling efficiency CSV found for group %d at %s.", top_group, eff_path)

    # --- write Block 3 payload (called block3 in JSON) ---

    payload_block3 = {
        "thermal_group": int(top_group),
        "center": {
            "lat": float(center_lat),
            "lon": float(center_lon),
            "utm_zone": int(meta["utm_zone"]),
            "x_m_center": float(meta["x_center"]),
            "y_m_center": float(meta["y_center"]),
        },
        "ranges": {
            "axis_limit_m": float(axis_limit),
            "zmin_m": float(zmin),
            "zmax_m": float(zmax),
        },
        "block3": {
            "xy_slices_records": xy_slices_records,
            "vertical_profile": vertical_profile,
            "rolling_strength": {
                "altitude_profile_roll": alt_roll_records,
                "heading_profile_roll": heading_roll_records,
                "params": {
                    "alt_step_m": alt_step,
                    "alt_sigma_bins": alt_sigma,
                    "heading_step_deg": hdg_step,
                    "heading_sigma_bins": hdg_sigma_bins,
                },
                "plots": {
                    "rolling_profiles_html": roll_html,
                },
            },
            "plots": {
                "xy_slices_html": xy_html,
                "paths_3d_html": threeD_html,
                "mean_plane_html": avg_plane_html,
            },
                        "average_plane": {
                "mean_alt_m": float(mean_alt_plane),
                "radial_profile": avg_plane_profile,
                "radius_metrics": {
                    "spread_radius_m": float(spread_radius),
                    "radius_2sigma_m": float(radius_2sigma),
                    "sigma_r_m": float(sigma_r),
                    "R_ref_models_m": float(R_ref_models),
                },
                "model_comparison": {
                    "models": model_fits,
                    "profiles": model_profiles,
                },
                "optimal_thermalling": optimal_thermalling_json,
            },
        },
    }

    summary = json_merge_save(
        summary_json_path(top_group),
        payload_block3,
        reset=False,
    )
    print(f"Block 3 (strength distribution) stored → {summary_json_path(top_group)}")


In [ ]:
# Objective 5.4 : Strength Distribuition calculation on Optimum Thermalling Radius (Block 4)
# 1. Using median optimum radius from Block 3, and height range from Block 2, create a 3D shape of optimum thermalling
# 2. on this optimum radius, map the variation of TE rate vs altitude and heading
# 3. visualize the TE rate distribution on the thermal cylinder surface

def block4_cylinder_surface() -> None:
    global R_mean, z0, z1, Turbo, C_MIN, C_MAX  # if you need them later in Block 6

    with open(summary_json_path(top_group), "r", encoding="utf-8") as f:
        S = json.load(f)

    if "block2" in S and "geometry" in S["block2"]:
        z0 = float(S["block2"]["geometry"].get("z_start_m", S["ranges"]["zmin_m"]))
        z1 = float(S["block2"]["geometry"].get("z_end_m",   S["ranges"]["zmax_m"]))
        R_mean = float(S["block2"]["geometry"]["radius_mean_m"])
    else:
        z0 = float(S["ranges"]["zmin_m"]); z1 = float(S["ranges"]["zmax_m"])
        R_mean = float(np.nanmedian(S["block2"]["smooth_R_z"]["R_m"]))

    roll_alt = S["block3"]["rolling_strength"]["altitude_profile_roll"]  # [{alt_m, te_rate_roll}]
    z_src = np.array([r["alt_m"] for r in roll_alt], float)
    w_src = np.array([r["te_rate_roll"] for r in roll_alt], float)
    Zline = np.linspace(z0, z1, 120)
    W0 = np.interp(np.clip(Zline, z_src.min(), z_src.max()), z_src, w_src)
    W0 = np.clip(np.where(np.isfinite(W0), W0, 0.0), 0.0, None)

    n_theta = 120
    theta = np.linspace(0, 2*np.pi, n_theta)
    TT, ZZ = np.meshgrid(theta, Zline, indexing="ij")
    XX = R_mean * np.cos(TT)
    YY = R_mean * np.sin(TT)
    surface_color = np.tile(W0, (n_theta, 1))

    roll_hdg = S["block3"]["rolling_strength"]["heading_profile_roll"]  # [{heading_deg, te_rate_roll}]
    hdg_deg = np.array([r["heading_deg"] for r in roll_hdg], float) % 360.0
    whdg    = np.array([r["te_rate_roll"] for r in roll_hdg], float)
    theta_deg = np.linspace(0, 360, n_theta, endpoint=False)
    w_theta = np.interp(theta_deg, hdg_deg, whdg, period=360.0)
    m_theta = w_theta / max(np.mean(w_theta), 1e-9)
    m_theta = np.clip(m_theta, 0.5, 1.5)

    surface_color = np.outer(m_theta, W0)
    white_red = [[0.0,"#ffffff"],[1.0,"#d00000"]]
    fig = go.Figure()
    fig.add_trace(go.Surface(
        x=XX, y=YY, z=ZZ,
        surfacecolor=surface_color,
        colorscale=white_red,
        cmin=float(surface_color.min()), cmax=float(surface_color.max()),
        colorbar=dict(title="TE rate [J/kg/s]"),
        showscale=True, opacity=0.95, name="Thermal cylinder"
    ))
    fig.add_trace(go.Scatter3d(
        x=[0,0], y=[0,0], z=[z0, z1],
        mode="lines", line=dict(width=6, dash="dash"), name="Center"
    ))
    fig.update_layout(
        scene=dict(xaxis_title="East (m)", yaxis_title="North (m)", zaxis_title="Altitude (m)"),
        title="Strength Distribuition visualization on Optimum Thermalling Radius",
        height=900, showlegend=True
    )

    out_html = None
    if PLOT_THERMAL_CYLINDER:
        out_html = (
            f"Results/plots/thermal_cylinder/"
            f"thermal_group_{top_group}_thermal_cylinder.html"
        )
        save_html(fig, out_html, label="Thermal cylinder (TE rate)")


In [ ]:
# Objective 5.5 : Strength Distribuition in Optimum Thermalling Radius along with Flight Paths (Block 6)
# 1. Load drift path, cylinder geometry, and TE profiles; set a unified time origin.
# 2. Load flight paths, convert to UTM coordinates consistent with drift path.
# 3. Create thermal cylinder TE-rate field vs altitude and heading, clipped to unified range.

def block6_animated_paths() -> None:
    global df_gp, t_path, xc_path, yc_path, XX0, YY0, ZZ, surface_color0
    global TRAIL_SEC, FRAME_MS, TRANS_MS, TICK_SPACING, t0_abs, N_FRAMES, OPACITY_CYL
    global Turbo, C_MIN, C_MAX, R_mean, z0, z1


    N_THETA   = 60
    N_Z       = 40
    N_FRAMES  = 60
    TRAIL_SEC = 120.0
    OPACITY_CYL = 0.75
    FRAME_MS = 200
    TRANS_MS = 0
    TICK_SPACING = 200

    # unified TE-rate colour scale and range (used for both model + flight path)
    Turbo = [
        [0.00, "#0000ff"],  # blue   = sink
        [0.33, "#00ff00"],  # green  = weak / no lift
        [0.66, "#ffff00"],  # yellow = moderate lift
        [1.00, "#ff0000"],  # red    = strong lift
    ]
    C_MIN = -5           # downdraft limit
    C_MAX = 30            # strong lift limit

    with open(summary_json_path(top_group), "r", encoding="utf-8") as f:
        S = json.load(f)

    if "block2" not in S or "drift" not in S["block2"]:
        raise ValueError("Missing drift path in summary JSON: expected block2.drift")
    geom = S["block2"].get("geometry", {})
    R_mean = float(geom.get("radius_mean_m", 300.0))
    z0 = float(geom.get("z_start_m", S["ranges"]["zmin_m"]))
    z1 = float(geom.get("z_end_m",   S["ranges"]["zmax_m"]))

    dr = S["block2"]["drift"]["drift_path_xy_m"]
    t_abs = pd.to_datetime(pd.Series(dr["t_hms"]), format="%H:%M:%S")

    # unified time origin: first drift timestamp
    t0_abs = t_abs.iloc[0]

    t_path = (t_abs - t0_abs).dt.total_seconds().to_numpy()
    xc_path = np.asarray(dr["x_m"], dtype=float)
    yc_path = np.asarray(dr["y_m"], dtype=float)

    roll_alt = S["block3"]["rolling_strength"]["altitude_profile_roll"]
    z_src = np.array([r["alt_m"] for r in roll_alt], float)
    w_src = np.array([r["te_rate_roll"] for r in roll_alt], float)

    roll_hdg = S["block3"]["rolling_strength"]["heading_profile_roll"]
    hdg_deg  = np.array([r["heading_deg"] for r in roll_hdg], float)
    whdg     = np.array([r["te_rate_roll"] for r in roll_hdg], float)

    theta = np.linspace(0, 360, N_THETA, endpoint=False)
    zline = np.linspace(z0, z1, N_Z)
    W0 = np.interp(np.clip(zline, z_src.min(), z_src.max()), z_src, w_src)
    W0 = np.clip(np.where(np.isfinite(W0), W0, 0.0), 0.0, None)

    w_theta = np.interp(theta, hdg_deg % 360.0, whdg, period=360.0)
    m_theta = w_theta / max(np.mean(w_theta), 1e-9)
    m_theta = np.clip(m_theta, 0.5, 1.5)

    TT, ZZ = np.meshgrid(np.deg2rad(theta), zline, indexing="ij")
    XX0 = R_mean * np.cos(TT)
    YY0 = R_mean * np.sin(TT)

    df_gp = df_top.copy()
    df_gp.dropna(subset=["lat","lon","alt","time_dt"], inplace=True)

    ref = S["block2"]["drift"]["drift_ref_latlon"]
    lat0 = float(S["center"]["lat"])
    lon0 = float(S["center"]["lon"])
    x_m, y_m, _ = utm_xy(df_gp["lat"].to_numpy(), df_gp["lon"].to_numpy(), lat0, lon0)
    df_gp["x_m"] = x_m
    df_gp["y_m"] = y_m
    xd_ref, yd_ref, _ = utm_xy(np.array([ref["lat0"]]), np.array([ref["lon0"]]), lat0, lon0)
    xc_path = xc_path + float(xd_ref[0])
    yc_path = yc_path + float(yd_ref[0])

    # flight times also relative to t0_abs (first drift time)
    df_gp["t_s"] = (df_gp["time_dt"] - t0_abs).dt.total_seconds().astype(float)

    # model cylinder TE-rate field, clipped to unified range
    surface_color0 = np.outer(m_theta, W0)
    surface_color0 = np.clip(surface_color0, C_MIN, C_MAX)

    MARGIN = 100.0
    X_MIN = float(min(df_gp["x_m"].min(), xc_path.min() - R_mean) - MARGIN)
    X_MAX = float(max(df_gp["x_m"].max(), xc_path.max() + R_mean) + MARGIN)
    Y_MIN = float(min(df_gp["y_m"].min(), yc_path.min() - R_mean) - MARGIN)
    Y_MAX = float(max(df_gp["y_m"].max(), yc_path.max() + R_mean) + MARGIN)
    Z_MIN = float(min(z0, df_gp["alt"].min()))
    Z_MAX = float(max(z1, df_gp["alt"].max()))
    RX = X_MAX - X_MIN
    RY = Y_MAX - Y_MIN
    RZ = Z_MAX - Z_MIN
    S  = max(RX, RY, RZ)

    t_min = max(t_path.min(), df_gp["t_s"].min())
    t_max = min(t_path.max(), df_gp["t_s"].max())
    t_frames = np.linspace(t_min, t_max, N_FRAMES)

    def hms(sec: float) -> str:
        return (t0_abs + pd.to_timedelta(sec, unit="s")).strftime("%H:%M:%S")

    frames = []
    for tf in t_frames:
        xc = float(np.interp(tf, t_path, xc_path))
        yc = float(np.interp(tf, t_path, yc_path))
        XX = XX0 + xc
        YY = YY0 + yc
        mask = (df_gp["t_s"] >= tf - TRAIL_SEC) & (df_gp["t_s"] <= tf)
        g = df_gp.loc[mask]
        frames.append(go.Frame(
            name=hms(tf),
            data=[
                go.Surface(
                    x=XX, y=YY, z=ZZ,
                    surfacecolor=surface_color0,
                    colorscale=Turbo,
                    cmin=C_MIN, cmax=C_MAX,
                    showscale=True,
                    opacity=OPACITY_CYL
                ),
                go.Scatter3d(
                    x=g["x_m"], y=g["y_m"], z=g["alt"],
                    mode="markers",
                    marker=dict(
                        size=3,
                        color=g["TotalEnergy_rate"].clip(C_MIN, C_MAX),
                        colorscale=Turbo,
                        cmin=C_MIN, cmax=C_MAX
                    ),
                    showlegend=False
                ),
                go.Scatter3d(
                    x=[xc], y=[yc], z=[0.5*(z0+z1)],
                    mode="markers",
                    marker=dict(size=5, symbol="x", color="black"),
                    showlegend=False
                )
            ]
        ))

    tf0 = t_frames[0]
    xc0 = float(np.interp(tf0, t_path, xc_path))
    yc0 = float(np.interp(tf0, t_path, yc_path))
    XXi = XX0 + xc0
    YYi = YY0 + yc0
    mask0 = (df_gp["t_s"] >= tf0 - TRAIL_SEC) & (df_gp["t_s"] <= tf0)
    g0 = df_gp.loc[mask0]

    fig = go.Figure(
        data=[
            go.Surface(
                x=XXi, y=YYi, z=ZZ,
                surfacecolor=surface_color0,
                colorscale=Turbo,
                cmin=C_MIN, cmax=C_MAX,
                colorbar=dict(
                    title="TE rate [J/kg/s]",
                    tickmode="array",
                    tickvals=[-5, 0, 5, 10, 15, 20, 25],
                    ticktext=[
                        "−5 (sink)",
                        "0 (neutral)",
                        "5",
                        "10",
                        "15",
                        "20",
                        "35 (strong lift)"
                    ]
                ),
                showscale=True,
                opacity=OPACITY_CYL,
                name="Thermal cylinder",
                showlegend=False
            ),
            go.Scatter3d(
                x=g0["x_m"], y=g0["y_m"], z=g0["alt"],
                mode="markers",
                marker=dict(
                    size=3,
                    color=g0["TotalEnergy_rate"].clip(C_MIN, C_MAX),
                    colorscale=Turbo,
                    cmin=C_MIN, cmax=C_MAX
                ),
                showlegend=False
            ),
            go.Scatter3d(
                x=[xc0], y=[yc0], z=[0.5*(z0+z1)],
                mode="markers",
                marker=dict(size=5, symbol="x", color="black"),
                showlegend=False
            )
        ],
        layout=go.Layout(
            title=f"Group {top_group} — Strength Distribuition in Optimum Thermalling Radius along with Flight Paths ",
            scene=dict(
                domain=dict(x=[0.0, 1.0], y=[0.12, 1.0]),
                xaxis=dict(
                    title="East (m)", range=[X_MIN, X_MAX],
                    tick0=np.floor(X_MIN / TICK_SPACING) * TICK_SPACING,
                    dtick=TICK_SPACING, showgrid=True
                ),
                yaxis=dict(
                    title="North (m)", range=[Y_MIN, Y_MAX],
                    tick0=np.floor(Y_MIN / TICK_SPACING) * TICK_SPACING,
                    dtick=TICK_SPACING, showgrid=True
                ),
                zaxis=dict(title="Altitude (m)", range=[Z_MIN, Z_MAX], showgrid=True),
                aspectmode="manual",
                aspectratio=dict(x=RX/S, y=RY/S, z=RZ/S)
            ),
            uirevision="keep",
            showlegend=True,
            height=950,
            margin=dict(l=10, r=10, t=40, b=10),
            updatemenus=[dict(
                type="buttons", showactive=False,
                buttons=[
                    dict(
                        label="Play", method="animate",
                        args=[None, {
                            "frame": {"duration": FRAME_MS, "redraw": True},
                            "transition": {"duration": TRANS_MS},
                            "fromcurrent": True, "mode": "immediate"
                        }]
                    ),
                    dict(
                        label="Pause", method="animate",
                        args=[[None], {
                            "frame": {"duration": 0, "redraw": False},
                            "transition": {"duration": 0},
                            "mode": "immediate"
                        }]
                    ),
                ]
            )],
            sliders=[dict(
                len=0.45, x=0.5, xanchor="center", y=0.02, yanchor="bottom",
                pad=dict(t=0, b=0),
                currentvalue=dict(prefix="", font=dict(size=1)),
                transition=dict(duration=0),
                steps=[
                    dict(
                        method="animate",
                        args=[[fr.name], {
                            "mode": "immediate",
                            "frame": {"duration": 0, "redraw": True},
                            "transition": {"duration": 0}
                        }],
                        label=""
                    ) for fr in frames
                ],
            )]
        ),
        frames=frames
    )
    fig.update_layout(margin=dict(l=10, r=10, t=40, b=10))

    out_html = None
    if PLOT_ANIMATED_CYLINDER_PATHS:
        out_html = (
            f"Results/plots/animated_cylinder_paths/"
            f"thermal_group_{top_group}_animated_cylinder_paths.html"
        )
        save_html(fig, out_html, label="Animated cylinder paths")


In [ ]:
# Objective 5.6 : Remove drift from Strength distribuition in Optimum Thermalling Radius along with Flight Paths (Block 7)

def block7_drift_free_paths() -> None:

    # reuse: Turbo, C_MIN, C_MAX, R_mean, z0, z1, t_path, xc_path, yc_path,
    #        df_gp (with x_m, y_m, alt, t_s, TotalEnergy_rate), XX0, YY0, ZZ,
    #        surface_color0, TRAIL_SEC, FRAME_MS, TRANS_MS, TICK_SPACING, t0_abs

    # transform flight path into thermal-core frame (subtract drift at each sample time)
    df_rel = df_gp.copy()

    df_rel["xc"] = np.interp(df_rel["t_s"], t_path, xc_path)
    df_rel["yc"] = np.interp(df_rel["t_s"], t_path, yc_path)

    df_rel["x_core"] = df_rel["x_m"] - df_rel["xc"]
    df_rel["y_core"] = df_rel["y_m"] - df_rel["yc"]

    MARGIN = 50.0

    X_CORE_MIN = float(df_rel["x_core"].min() - MARGIN)
    X_CORE_MAX = float(df_rel["x_core"].max() + MARGIN)
    Y_CORE_MIN = float(df_rel["y_core"].min() - MARGIN)
    Y_CORE_MAX = float(df_rel["y_core"].max() + MARGIN)

    Z_CORE_MIN = float(min(z0, df_rel["alt"].min()))
    Z_CORE_MAX = float(max(z1, df_rel["alt"].max()))

    RXc = X_CORE_MAX - X_CORE_MIN
    RYc = Y_CORE_MAX - Y_CORE_MIN
    RZc = Z_CORE_MAX - Z_CORE_MIN
    Sc  = max(RXc, RYc, RZc)

    # time window where both drift and glider data exist
    t_min_core = max(t_path.min(), df_rel["t_s"].min())
    t_max_core = min(t_path.max(), df_rel["t_s"].max())
    t_frames_core = np.linspace(t_min_core, t_max_core, N_FRAMES)

    def hms_core(sec: float) -> str:
        return (t0_abs + pd.to_timedelta(sec, unit="s")).strftime("%H:%M:%S")

    frames_core = []
    for tf in t_frames_core:
        mask = (df_rel["t_s"] >= tf - TRAIL_SEC) & (df_rel["t_s"] <= tf)
        g = df_rel.loc[mask]

        frames_core.append(
            go.Frame(
                name=hms_core(tf),
                data=[
                    # stationary thermal cylinder at origin
                    go.Surface(
                        x=XX0,
                        y=YY0,
                        z=ZZ,
                        surfacecolor=surface_color0,
                        colorscale=Turbo,
                        cmin=C_MIN,
                        cmax=C_MAX,
                        showscale=False,
                        opacity=OPACITY_CYL,
                    ),
                    # pilot points in thermal frame
                    go.Scatter3d(
                        x=g["x_core"],
                        y=g["y_core"],
                        z=g["alt"],
                        mode="markers",
                        marker=dict(
                            size=3,
                            color=g["TotalEnergy_rate"].clip(C_MIN, C_MAX),
                            colorscale=Turbo,
                            cmin=C_MIN,
                            cmax=C_MAX,
                        ),
                        showlegend=False,
                    ),
                    # mark thermal center at origin
                    go.Scatter3d(
                        x=[0.0],
                        y=[0.0],
                        z=[0.5 * (z0 + z1)],
                        mode="markers",
                        marker=dict(size=6, symbol="x", color="black"),
                        showlegend=False,
                    ),
                ],
            )
        )

    # initial frame
    tf0 = t_frames_core[0]
    mask0 = (df_rel["t_s"] >= tf0 - TRAIL_SEC) & (df_rel["t_s"] <= tf0)
    g0 = df_rel.loc[mask0]

    fig_core = go.Figure(
        data=[
            go.Surface(
                x=XX0,
                y=YY0,
                z=ZZ,
                surfacecolor=surface_color0,
                colorscale=Turbo,
                cmin=C_MIN,
                cmax=C_MAX,
                colorbar=dict(
                    title="TE rate [J/kg/s]",
                    tickmode="array",
                    tickvals=[-1.5, -0.5, 0.5, 1.5, 2.5, 3.5],
                    ticktext=[
                        "Downdraft (−2..−1)",
                        "Sink / neutral (−1..0)",
                        "No lift (0..1)",
                        "Weak lift (1..2)",
                        "Moderate (2..3)",
                        "Good / strong (3..4)",
                    ],
                ),
                showscale=True,
                opacity=OPACITY_CYL,
                name="Thermal cylinder",
                showlegend=False,
            ),
            go.Scatter3d(
                x=g0["x_core"],
                y=g0["y_core"],
                z=g0["alt"],
                mode="markers",
                marker=dict(
                    size=3,
                    color=g0["TotalEnergy_rate"].clip(C_MIN, C_MAX),
                    colorscale=Turbo,
                    cmin=C_MIN,
                    cmax=C_MAX,
                ),
                showlegend=False,
            ),
            go.Scatter3d(
                x=[0.0],
                y=[0.0],
                z=[0.5 * (z0 + z1)],
                mode="markers",
                marker=dict(size=6, symbol="x", color="black"),
                showlegend=False,
            ),
        ],
        layout=go.Layout(
            title=f"Group {top_group} — Drift-free frame (thermal fixed, pilots circling)",
            scene=dict(
                domain=dict(x=[0.0, 1.0], y=[0.12, 1.0]),
                xaxis=dict(
                    title="East (m, core frame)",
                    range=[X_CORE_MIN, X_CORE_MAX],
                    tick0=np.floor(X_CORE_MIN / TICK_SPACING) * TICK_SPACING,
                    dtick=TICK_SPACING,
                    showgrid=True,
                ),
                yaxis=dict(
                    title="North (m, core frame)",
                    range=[Y_CORE_MIN, Y_CORE_MAX],
                    tick0=np.floor(Y_CORE_MIN / TICK_SPACING) * TICK_SPACING,
                    dtick=TICK_SPACING,
                    showgrid=True,
                ),
                zaxis=dict(
                    title="Altitude (m)",
                    range=[Z_CORE_MIN, Z_CORE_MAX],
                    showgrid=True,
                ),
                aspectmode="manual",
                aspectratio=dict(x=RXc / Sc, y=RYc / Sc, z=RZc / Sc),
            ),
            uirevision="keep",
            showlegend=False,
            height=950,
            margin=dict(l=10, r=10, t=40, b=10),
            updatemenus=[
                dict(
                    type="buttons",
                    showactive=False,
                    buttons=[
                        dict(
                            label="Play",
                            method="animate",
                            args=[
                                None,
                                dict(
                                    frame=dict(duration=FRAME_MS, redraw=True),
                                    transition=dict(duration=TRANS_MS),
                                    fromcurrent=True,
                                    mode="immediate",
                                ),
                            ],
                        ),
                        dict(
                            label="Pause",
                            method="animate",
                            args=[
                                [None],
                                dict(
                                    frame=dict(duration=0, redraw=False),
                                    transition=dict(duration=0),
                                    mode="immediate",
                                ),
                            ],
                        ),
                    ],
                )
            ],
            sliders=[
                dict(
                    len=0.45,
                    x=0.5,
                    xanchor="center",
                    y=0.02,
                    yanchor="bottom",
                    pad=dict(t=0, b=0),
                    currentvalue=dict(prefix="", font=dict(size=1)),
                    transition=dict(duration=0),
                    steps=[
                        dict(
                            method="animate",
                            args=[
                                [fr.name],
                                dict(
                                    mode="immediate",
                                    frame=dict(duration=0, redraw=True),
                                    transition=dict(duration=0),
                                ),
                            ],
                            label="",
                        )
                        for fr in frames_core
                    ],
                )
            ],
        ),
        frames=frames_core,
    )

    fig_core.update_layout(margin=dict(l=10, r=10, t=40, b=10))

    out_html_core = None
    if PLOT_ANIMATED_CYLINDER_DRIFT_FREE:
        out_html_core = (
            f"Results/plots/animated_cylinder_paths_drift_free/"
            f"thermal_group_{top_group}_animated_cylinder_paths_drift_free.html"
        )
        save_html(fig_core, out_html_core, label="Animated cylinder paths (drift-free)")



In [ ]:
# Objective 5.7 : Run full modelling pipeline for a given thermal group

def run_modelling_for_group(group_id: int) -> None:
    global top_group, df_top
    top_group = int(group_id)

    df_top = (
        thermals_ds[thermals_ds["thermal_group"] == top_group]
        .sort_values(["Filename", "ThermalNumber", "flighttime"])
        .copy()
    )
    if df_top.empty:
        logger.warning("No rows for thermal_group = %d; skipping.", top_group)
        return


    print("\n" + "=" * 60)
    print(f"Running modelling for thermal_group = {top_group}")
    print("=" * 60)

    try:
        block2_opt_radius_and_drift()
        block3_strength_distribution()
        block4_cylinder_surface()
        block6_animated_paths()
        block7_drift_free_paths()
    except ValueError as e:
        logger.warning("Skipping thermal_group %d: %s", top_group, e)
        return

    print(f"Finished thermal_group = {top_group}\n")

if RUN_MODELLING_LOOPS:
    for g in modelling_groups:
        run_modelling_for_group(g)
else:
    run_modelling_for_group(top_group)

'''# condensed output
    if PLOT_LOG:
        # one line with plot locations
        print(f"Thermal group {top_group}: {len(PLOT_LOG)} plots saved -> {', '.join(PLOT_LOG)}")
    else:
        print(f"Thermal group {top_group}: no plots generated")

    # final single line for the group
    print(f"Thermal group {top_group} done")
'''


## 6. Summarization of all thermal groups

In [ ]:
# Objective 6.1 : Summarize modelling results for all thermal groups

print("\n=== Modelling summary for all thermal groups (no plots) ===")

df_model = thermals_ds.copy()
df_model["te_rate"] = choose_te_rate(df_model)

groups  = sorted(df_model["thermal_group"].dropna().unique())
records = []

for g in groups:
    df_g = df_model[df_model["thermal_group"] == g].copy()
    if df_g.empty:
        continue

    summary_path = summary_json_path(g)
    summary = {}
    if os.path.exists(summary_path):
        try:
            with open(summary_path, "r", encoding="utf-8") as f:
                summary = json.load(f)
        except Exception:
            summary = {}

    b2   = summary.get("block2") or {}
    geom = b2.get("geometry") or {}

    z_start_m = float(geom.get("z_start_m", df_g["alt"].quantile(0.02)))
    z_end_m   = float(geom.get("z_end_m",   df_g["alt"].quantile(0.98)))
    column_height_m = z_end_m - z_start_m

    opt_radius_for_thermalling_m, max_efficiency_achieved_pct = extract_opt_radius_and_max_efficiency(g)

    b3          = summary.get("block3") or {}
    avg_plane   = b3.get("average_plane") or {}
    rad_metrics = avg_plane.get("radius_metrics") or {}
    radius_2sigma_m = rad_metrics.get("radius_2sigma_m")
    radius_2sigma_m = float(radius_2sigma_m) if radius_2sigma_m is not None else np.nan

    model_info  = avg_plane.get("model_comparison") or {}
    models_list = model_info.get("models") or []
    closest_model = None
    closest_model_match_pct = np.nan
    if models_list:
        best = max(models_list, key=lambda m: m.get("match_percent", -np.inf))
        if np.isfinite(best.get("match_percent", np.nan)):
            closest_model = best.get("model")
            closest_model_match_pct = float(best.get("match_percent"))

    drift_speed_mps = np.nan
    if "time_dt" in df_g.columns and df_g["time_dt"].notna().any():
        g2 = df_g.dropna(subset=["time_dt", "lat", "lon"]).copy()
        if not g2.empty:
            g2["t_s"] = (g2["time_dt"] - g2["time_dt"].min()).dt.total_seconds()
            dt_bin    = 10.0
            t_bins    = np.arange(g2["t_s"].min(), g2["t_s"].max() + dt_bin, dt_bin)
            g2["t_bin"] = np.digitize(g2["t_s"], t_bins)

            centers = (
                g2.groupby("t_bin", as_index=False)
                  .agg(t_s=("t_s","mean"), lat=("lat","mean"), lon=("lon","mean"))
                  .dropna()
                  .sort_values("t_s")
            )
            if len(centers) >= 2:
                lat0 = float(centers["lat"].mean())
                lon0 = float(centers["lon"].mean())
                x, y, _ = utm_xy(
                    centers["lat"].to_numpy(),
                    centers["lon"].to_numpy(),
                    lat0,
                    lon0,
                )
                dist = float(np.sqrt(np.diff(x)**2 + np.diff(y)**2).sum())
                dt   = float(centers["t_s"].iloc[-1] - centers["t_s"].iloc[0])
                if dt > 0.0:
                    drift_speed_mps = dist / dt

    records.append({
        "thermal_group"                : int(g),
        "column_height_m"              : column_height_m,
        "opt_radius_for_thermalling_m" : opt_radius_for_thermalling_m,
        "radius_2sigma_m"              : radius_2sigma_m,
        "closest_model"                : closest_model,
        "closest_model_match_pct"      : closest_model_match_pct,
        "drift_speed_mps"              : drift_speed_mps,
        "max_efficiency_achieved_pct"  : max_efficiency_achieved_pct,
    })

geom_df = pd.DataFrame.from_records(records)

# merge into existing cluster_strength table
if "cluster_strength" in globals():
    if "te_gain_mean" in cluster_strength.columns:
        cluster_strength = cluster_strength.rename(columns={"te_gain_mean": "te_gain_mean_Jpkg"})
    if "radius_mean_m" in cluster_strength.columns:
        cluster_strength = cluster_strength.rename(columns={"radius_mean_m": "mean_radius_of_pilots_m"})
    else:
        logger.warning("mean_radius_of_pilots_m not found in cluster_strength")

    # drop only columns we are about to overwrite
    cols_to_drop = [c for c in geom_df.columns if c in cluster_strength.columns and c != "thermal_group"]
    if cols_to_drop:
        cluster_strength = cluster_strength.drop(columns=cols_to_drop)

    cluster_strength = cluster_strength.merge(geom_df, on="thermal_group", how="left")

    print("\nUpdated cluster_strength with modelling metrics:")

    sort_cols = [c for c in ["n_thermals", "te_gain_mean_Jpkg"] if c in cluster_strength.columns]
    cluster_strength_sorted = cluster_strength.sort_values(sort_cols, ascending=[False] * len(sort_cols))

    cols = [
        "thermal_group",
        "n_pilots",
        "n_thermals",
        "te_gain_mean_Jpkg",
        "column_height_m",
        "mean_radius_of_pilots_m",
        "opt_radius_for_thermalling_m",
        "radius_2sigma_m",
        "closest_model",
        "closest_model_match_pct",
        "drift_speed_mps",
        "max_efficiency_achieved_pct",
    ]
    cols = [c for c in cols if c in cluster_strength_sorted.columns]

    print(cluster_strength_sorted[cols].to_string(index=False))
else:
    logger.warning("cluster_strength not in scope; only geom_df created.")

# ---------------- EXPORT FINAL SUMMARY ----------------
out_csv = RESULTS_DIR / "Final_summary_of_thermal_groups.csv"

cluster_strength_sorted[cols].to_csv(out_csv, index=False)

print(f"\nSaved final summary to: {out_csv}")
